In [1]:
from fastapi import APIRouter, Query
from typing import Optional, List, Dict, Any
from datetime import datetime, timedelta
from collections import defaultdict
import pandas as pd
import json
import re
import os
import numpy as np
os.chdir('D:\A0_Project\AOI_Density')
from models.sql_db_connect import MySQLConnet
dbhandler = MySQLConnet('l6a01_project')

import warnings
warnings.filterwarnings('ignore')

In [3]:
class Config:
    def __init__(self):
        # ================================================= datetime =================================================
        now = datetime.now()
        ym = now.strftime("%Y%m")
        print(f"連線時間: {now}({ym})")
        one_mon_ago = now - timedelta(days=30)

        # ================================================= 資料設定  =================================================
        # === PI Line & AOI Tool  ====
        self.uni_aoi_names = [f'aoi{i}00' for i in range(1,4,1)]
        self.uni_pi_names = [f'pi{i}00' for i in range(1,8,1)]

        # === particle type & defect code ====
        self.uni_UPI_defect_codes = ['Polymer', 'SSIU_Polymer']
        self.uni_SPOT_defect_codes = ['PI_Spot_NP', 'PIS With Particle']
        self.uni_SPS_defect_codes = ['SPS']
        self.uni_defect_types = ['Particle', 'PISpot']
        self.all_defect_codes = ['Polymer', 'SSIU_Polymer', 'PI_Spot_NP', 'PIS With Particle', 'SPS']

        # ==== defect size config  ===
        self.uni_defect_sizes = ['S', 'M', 'L', 'O']
        self.rawdata_defect_size_col = 'defect_size'
        self.size_group_keys = ['S','M','L','O','SM','SL','SO','ML','MO','LO','SML','SMO','SLO','MLO','SMLO']
        self.defect_size_rules = {
            "S": lambda x: x <= 20,
            "M": lambda x: 21 <= x <= 100,
            "L": lambda x: 101 <= x <= 400,
            "O": lambda x: x >= 401,
        }

        # === Glass side  ===
        self.glass_sides = ['CF', 'TFT']

        # =================================================  UPI/SPOT Default config  =================================================
        self.tab_filter_config = {
            'UPI': {
                'line_id': [f'CAPIC{i}00' for i in range(1,8,1)],#['CAPIC200'],
                'ai_code_1': self.uni_UPI_defect_codes,
                'recipe_id':[]
            },
            'PISpot': {
                'line_id': [f'CAPIC{i}00' for i in range(1,8,1)],#['CAPIC200'],
                'ai_code_1': self.uni_SPOT_defect_codes,
                'recipe_id':[]
            },
            'SPS': {
                'line_id': [f'CAPIC{i}00' for i in range(1,8,1)],
                'ai_code_1': self.uni_SPS_defect_codes,
                'recipe_id':[]
            },
            'SPEC':{}
        }

        # ================================================= MySQL AOI Density 資料表設定 =================================================
        self.aoi_pidensit_summary_tbn = f'pidenisty_pihour_{ym}'
        self.aoi_pi_density_tbns = [f'{aoi}_pidensity_yyyymm_{pi}' for aoi in self.uni_aoi_names for pi in self.uni_pi_names]
        self.aoi_pidensity_spec_tbn = 'aoi_spec_for_aoimonitor'
        print('預設讀取:', self.aoi_pidensit_summary_tbn)

        self.aoi_pidensity_spec_cols = ['NO', 'MODEL_ID', 'MODEL_TYPE', 'DEFECT_CODE', 'PROCESS_TYPE', 'SIZE_TYPE', 'OOC', 'OOS']
        self.aoi_density_summary_sql_cols = [
            'scan_time', 'pi_time', 'pi_hour', 'aoi', 'line_id', 'model',
            'glass_type', 'recipe_id', 'ai_code_1', 'pic_paths',
            'maingroup_glass_count', 'maingroup_defect_count',
            'defect_code_glass_count', 'defect_code_count',
            'small_defect_count', 'middle_defect_count', 'large_defect_count', 'over_defect_count',
            'glass', 'noteText'
        ]

        self.aoi_density_rawdata_sql_cols = [
            'scan_time', 'line_id', 'model', 'glass_type', 'recipe_id', 'glass_id',
            'pic_name', 'x', 'y', 'predict_code', 'judge_code', 'mark', 'hour','dayhour', 'day', 'year', 'month', 'season', 'week', 'yearmonth',
            'defect_count', 'defect_size', 'open_status', 'ai_code_1', 'ai_code_2',
            'ai_code_3', 'ai_code_4', 'ai_code_5', 'gray_name', 'ip_num',
            'first_code', 'chip_name', 'defect_seq', 'pi_time', 'pi_hour',
            'pic_path', 'recipe_comment'
        ]

        
        self.PRIMARY_GROUP_COLS = ["pi_hour", "line_id", "model", "glass_type", "recipe_id"]
        # ================================================= 前端網頁CONFIG =================================================
        self.chart_table_coldict = {
            'line_id': 'PI Line',
            'aoi': 'aoi',
            'model': 'Model',
            'glass_type': 'side',
            'pi_hour': 'Hourly',
            'recipe_id': 'recipe',
            'maingroup_glass_count': 'total gld',   # maingroup分群後的玻璃總片數
            'ai_code_1': 'defect',
            #'maingroup_defect_count': 'total def',  # maingroup分群後的總defect 數
            'glass': 'glass',                        # list ,次分群中對應的glass名稱
            #'defect_code_glass_count': 'gld',  # 主群後再分ai_code_1的玻璃數
            'defect_code_count': 'def count ',        # 主群後再分ai_code_1的defect數
            
            'glass_defect_count': 'size' # dsict, 單片glass defect
        }

        self.table_group_key_dict = {
            'main_group': ['pi_hour', 'line_id', 'aoi', 'model', 'recipe_id', 'glass_type', 'maingroup_glass_count', 'ai_code_1'],
                           #'maingroup_glass_count', 'maingroup_defect_count'],
            'uni_col': ['defect_code_glass_count', 'defect_code_count', 'glass_defect_count', 'glass']
            # glass 依照現有邏輯分割後單獨對應 uni_col 中的其他欄位資訊
        }

        self.chart_group_dict = {
            'left': ['line_id','model','maingroup_glass_count', 'defect_code_glass_count'],
            'up':   ['aoi', 'ai_code_1'],
            'down': ['pi_hour'],
            'right':['density'],
        }

        self.uni_glass_row_info_dict = {
            'glass_id': 'glass',
            'glass_defect_count':'glass_defect_count',
            'small_defect_count': 'S',
            'middle_defect_count': 'M',
            'large_defect_count': 'L',
            'over_defect_count':  'O'
        }

        self.defect_group_coldict = {'x': 'x', 'y': 'y', 'chip_name': 'chip', 'pic_name':'img'}

        self.filter_item_coldict = {
            'line_id': 'PI Line',
            'aoi': 'aoi tools',
            'model': 'Model',
            'recipe_id': 'recipe',
            'glass_type': 'glass_side',
            'ai_code_1': 'defect code',
            'defect_size': 'defect size'
        }

        self.front_config = {
            'chartKeyDict': self.chart_group_dict,
            'filtetItemKeyDict': self.filter_item_coldict,
            'hourlyTable': self.chart_table_coldict,
            'hourlyTable_key_group': self.table_group_key_dict,
            'uniGlassInfo': self.uni_glass_row_info_dict,
            'uniGlassDefectTable': self.defect_group_coldict,
            'SubTabsFilterDefaultDict': self.tab_filter_config
        }

# -------- 小工具 --------

cfg = Config()

連線時間: 2025-11-27 13:09:24.656093(202511)
預設讀取: pidenisty_pihour_202511


In [ ]:
spec_df = dbhandler.get_table(cfg.aoi_pidensity_spec_tbn)
spec_df.iloc[0,:].to_dict()

default_spec_cols_dict = {
    'NO': 'NO',
 'MODEL_ID': 'model',
 'MODEL_TYPE': 'MODEL_TYPE',
 'DEFECT_CODE': 'ai_code_1',
 'PROCESS_TYPE': 'PROCESS_TYPE',
 'SIZE_TYPE': 'defect_size',
 'OOC': 'OOC',
 'OOS': 'OOS'}

[]

{0: {'NO': '1',
  'MODEL_ID': 'B070AAN01',
  'MODEL_TYPE': 'Normal',
  'DEFECT_CODE': 'PI_Spot_NP',
  'PROCESS_TYPE': 'BIG',
  'SIZE_TYPE': 'OL',
  'OOC': '0.33',
  'OOS': '0.57'},
 1: {'NO': '1',
  'MODEL_ID': 'B070AAN01',
  'MODEL_TYPE': 'Normal',
  'DEFECT_CODE': 'PI_Spot_NP',
  'PROCESS_TYPE': 'BIG',
  'SIZE_TYPE': 'OLMS',
  'OOC': '300',
  'OOS': '300'},
 2: {'NO': '1',
  'MODEL_ID': 'B070AAN01',
  'MODEL_TYPE': 'Normal',
  'DEFECT_CODE': 'PIS With Particle',
  'PROCESS_TYPE': 'BIG',
  'SIZE_TYPE': 'OL',
  'OOC': '0.28',
  'OOS': '0.33'},
 3: {'NO': '1',
  'MODEL_ID': 'B070AAN01',
  'MODEL_TYPE': 'Normal',
  'DEFECT_CODE': 'PIS With Particle',
  'PROCESS_TYPE': 'BIG',
  'SIZE_TYPE': 'OLMS',
  'OOC': '2',
  'OOS': '3'},
 4: {'NO': '1',
  'MODEL_ID': 'B070AAN01',
  'MODEL_TYPE': 'Normal',
  'DEFECT_CODE': 'Polymer',
  'PROCESS_TYPE': 'BIG',
  'SIZE_TYPE': 'OLMS',
  'OOC': '5',
  'OOS': '10'},
 5: {'NO': '1',
  'MODEL_ID': 'B070AAN01',
  'MODEL_TYPE': 'Normal',
  'DEFECT_CODE': 'SSIU

In [3]:
#cfg.aoi_pi_density_tbns
tbns = dbhandler.list_tables()
tbns#[v for v in tbns if 'spec' in v]

2025-11-17 13:57:23,502 - INFO - [list_tables] 成功取得資料表名稱，共 84 張表。


['aoi100_glassdata_202511',
 'aoi100_pidensity_202509_pi000',
 'aoi100_pidensity_202509_pi100',
 'aoi100_pidensity_202509_pi200',
 'aoi100_pidensity_202509_pi300',
 'aoi100_pidensity_202509_pi400',
 'aoi100_pidensity_202509_pi500',
 'aoi100_pidensity_202509_pi600',
 'aoi100_pidensity_202509_pi700',
 'aoi100_pidensity_202510_pi000',
 'aoi100_pidensity_202510_pi100',
 'aoi100_pidensity_202510_pi200',
 'aoi100_pidensity_202510_pi300',
 'aoi100_pidensity_202510_pi400',
 'aoi100_pidensity_202510_pi500',
 'aoi100_pidensity_202510_pi600',
 'aoi100_pidensity_202510_pi700',
 'aoi100_pidensity_202511_pi000',
 'aoi100_pidensity_202511_pi100',
 'aoi100_pidensity_202511_pi200',
 'aoi100_pidensity_202511_pi300',
 'aoi100_pidensity_202511_pi400',
 'aoi100_pidensity_202511_pi500',
 'aoi100_pidensity_202511_pi600',
 'aoi100_pidensity_202511_pi700',
 'aoi100_rawdata_202508',
 'aoi100_rawdata_202509',
 'aoi100_rawdata_202510',
 'aoi100_rawdata_202511',
 'aoi200_glassdata_202511',
 'aoi200_pidensity_20250

In [ ]:
def drop_tb(drop_list):
  for tbn in drop_list:
    dbhandler.drop_table(tbn)
  
drop_list = ['pidenisty_pihour_202511']
#drop_tb(drop_list)

2025-11-11 14:02:36,624 - INFO - [drop_table] 資料表 'pidenisty_pihour_202511' 已刪除.


In [9]:
ORI_PIDENSITY = ['scan_time', 'line_id', 'model', 'glass_type', 'recipe_id', 'glass_id',
       'pic_name', 'x', 'y', 'predict_code', 'judge_code', 'mark', 'hour',
       'dayhour', 'day', 'year', 'month', 'season', 'week', 'yearmonth',
       'defect_count', 'defect_size', 'open_status', 'ai_code_1', 'ai_code_2',
       'ai_code_3', 'ai_code_4', 'ai_code_5', 'gray_name', 'ip_num',
       'first_code', 'chip_name', 'defect_seq', 'pi_time', 'pi_hour',
       'pic_path', 'recipe_comment']

USED_PIDENSITY = ['scan_time','pi_time', 'line_id','aoi', 'model', 'glass_type', 'recipe_id', 'glass_id',
       'pic_name', 'x', 'y', 'defect_count', 'defect_size', 'ai_code_1', 'pic_path']

USED_DEFECTMAP = ['scantime', 'line_id', 'model', 'glass_type', 'recipe_id', 'glass_id',
       'pic_name', 'x', 'y'
       'defect_count', 'defect_size', 'open_status', 'ai_code_1', 'ai_code_2',
       'ai_code_3', 'ai_code_4', 'ai_code_5', 'gray_name', 'ip_num',
       'first_code', 'chip_name', 'defect_seq', 'image_path',
       'over_defect_count', 'large_defect_count', 'middle_defect_count',
       'small_defect_count']

In [ ]:
{'scan_time': Timestamp('2025-11-06 14:04:31'),
 'pi_time': Timestamp('2025-11-06 10:24:24'),
 'line_id': 'CAPIC200',
 'aoi':'aoi100',
 'model': 'M270HAN1C',
 'glass_type': 'CF',
 'recipe_id': '105',
 'glass_id': 'AR0HJB05RF',
 'pic_name': 'CaptureImage/CAPIT203_AR0HJB05RF_10_16_2511061404.JPG',
 'x': '1781986',
 'y': '1419131',
 'defect_count': '16',
 'defect_size': '18',
 'ai_code_1': 'OP_Defect',
 'pic_path': 'http://10.97.139.98:1454//CAPIT203/CA0056/AR0HJB05RF/A/20251106140431/'}

In [11]:
cfg = Config()
tbn = 'his_90days_spec_table'
tbn = 'spec_pro_update__table'
tbn = 'aoi_spec_for_aoimonitor'
tbn = 'pidensity_202511'
tbn = 'pidenisty_pihour_202511'
tbn ='aoi100_pidensity_202511_pi200'
#tbn = 'aoi100_rawdata_202511'
tb = dbhandler.get_table(tbn)
print(tbn, len(tb))
"""
tb2 = tb[tb['ai_code_1'].isin(cfg.all_defect_codes)][[ 'pi_hour', 'aoi', 'line_id', 'model',
       'recipe_id', 'ai_code_1',
       'maingroup_glass_count',
       'maingroup_defect_count', 'defect_code_glass_count',
       'defect_code_count', 'glass']]"""
print(tb.columns)
tb[[cl for cl in USED_PIDENSITY if cl in tb.columns]].iloc[0,:].to_dict()

連線時間: 2025-11-13 15:01:25.379993(202511)
預設讀取: pidenisty_pihour_202511
aoi100_pidensity_202511_pi200 310
Index(['scan_time', 'line_id', 'model', 'glass_type', 'recipe_id', 'glass_id',
       'pic_name', 'x', 'y', 'predict_code', 'judge_code', 'mark', 'hour',
       'dayhour', 'day', 'year', 'month', 'season', 'week', 'yearmonth',
       'defect_count', 'defect_size', 'open_status', 'ai_code_1', 'ai_code_2',
       'ai_code_3', 'ai_code_4', 'ai_code_5', 'gray_name', 'ip_num',
       'first_code', 'chip_name', 'defect_seq', 'pi_time', 'pi_hour',
       'pic_path', 'recipe_comment'],
      dtype='object')


{'scan_time': Timestamp('2025-11-06 14:04:31'),
 'pi_time': '2025-11-06 10:24:24',
 'line_id': 'CAPIC200',
 'model': 'M270HAN1C',
 'glass_type': 'CF',
 'recipe_id': '105',
 'glass_id': 'AR0HJB05RF',
 'pic_name': 'CaptureImage/CAPIT203_AR0HJB05RF_10_16_2511061404.JPG',
 'x': '1781986',
 'y': '1419131',
 'defect_count': '16',
 'defect_size': '18',
 'ai_code_1': 'OP_Defect',
 'pic_path': 'http://10.97.139.98:1454//CAPIT203/CA0056/AR0HJB05RF/A/20251106140431/'}

In [19]:
def filter_flexible(df, kv: dict, *, na_equal: bool=False, casefold: bool=False):
    """
    kv 的 value 可以是：
      - 單一值（含字串）→ 以等號比對
      - list/tuple/set → 以 isin 比對
    參數：
      na_equal=True  時，允許把 None/NaN 視為相等（單值或清單均適用）
      casefold=True  時，字串欄位與字串值都做大小寫無關比對（含 isin 的清單字串）
    """
    m = pd.Series(True, index=df.index)

    for col, val in kv.items():
        s = df[col]

        # 大小寫無關：把字串欄位與字串值（或清單中的字串）都做 casefold
        if casefold and is_string_dtype(s):
            s_cmp = s.astype("string").str.casefold()
            if isinstance(val, (list, tuple, set)):
                vals = [v.casefold() if isinstance(v, str) else v for v in val]
            else:
                vals = val.casefold() if isinstance(val, str) else val
        else:
            s_cmp = s
            vals = list(val) if isinstance(val, (list, tuple, set)) else val

        if isinstance(vals, list):  # isin 路徑
            # 先把非 NA 目標值拿來做 isin
            non_na_targets = []
            has_na_target = False
            for v in vals:
                if na_equal and (v is None or (isinstance(v, float) and math.isnan(v))):
                    has_na_target = True
                else:
                    non_na_targets.append(v)

            mask = s_cmp.isin(non_na_targets) if non_na_targets else pd.Series(False, index=s.index)
            if na_equal and has_na_target:
                mask = mask | s.isna()  # 用原欄位判斷 NaN
            m &= mask

        else:  # 等號路徑
            if na_equal and (vals is None or (isinstance(vals, float) and math.isnan(vals))):
                m &= s.isna()
            else:
                m &= s_cmp.eq(vals)

    return df.loc[m]

In [39]:
PRIMARY_GROUP_COLS = [
    "scan_time", "pi_time", "pi_hour", "line_id",'aoi', "model",
    "glass_type", "recipe_id", "pic_path", "recipe_comment"
]

In [62]:
info_keys = ['pi_hour', 'aoi', 'line_id', 'model',
       'glass_type', 'recipe_id', 'ai_code_1',
        'maingroup_glass_count',
       'maingroup_defect_count', 'defect_code_glass_count',
       'defect_code_count', 'small_defect_count', 'middle_defect_count',
       'large_defect_count', 'over_defect_count', 'glass', 'noteText', 'recipe_comment', 'pic_path']

In [28]:
cfg= Config()
tbn = 'pidenisty_pihour_202511'
#tbn =  'aoi100_pidensity_202511_pi700'
df = dbhandler.get_table(tbn)
df

連線時間: 2025-11-17 14:18:47.038977(202511)
預設讀取: pidenisty_pihour_202511


,pi_hour,aoi,line_id,model,glass_type,recipe_id,ai_code_1,maingroup_glass_count,maingroup_defect_count,defect_code_glass_count,defect_code_count,small_defect_count,middle_defect_count,large_defect_count,over_defect_count,glass,pic_paths,noteText
0,2025-11-03 07:00:00,aoi200,CAPIC500,M270DAN8S,TFT,0286,Not_Found,4,421,4,17,8,9,0,0,"VQ5AANS8C,VQ5AANS8D,VQ5AANS8E,VQ5AANS8F",http://10.97.139.98:1454/CAAOI202/CA007001/JF5...,None
1,2025-11-03 07:00:00,aoi200,CAPIC500,M270DAN8S,TFT,0286,NPI_TFT,4,421,3,5,0,4,1,0,"VQ5AANS8C,VQ5AANS8D,VQ5AANS8E",http://10.97.139.98:1454/CAAOI202/CA007001/JF5...,None
2,2025-11-03 07:00:00,aoi200,CAPIC500,M270DAN8S,TFT,0286,PI_Gel,4,421,2,2,0,2,0,0,"VQ5AANS8C,VQ5AANS8D",http://10.97.139.98:1454/CAAOI202/CA007001/JF5...,None
3,2025-11-03 07:00:00,aoi200,CAPIC500,M270DAN8S,TFT,0286,PI_Spot_NP,4,421,4,358,33,319,6,0,"VQ5AANS8C,VQ5AANS8D,VQ5AANS8E,VQ5AANS8F",http://10.97.139.98:1454/CAAOI202/CA007001/JF5...,None
4,2025-11-03 07:00:00,aoi200,CAPIC500,M270DAN8S,TFT,0286,Polymer,4,421,3,4,0,4,0,0,"VQ5AANS8C,VQ5AANS8E,VQ5AANS8F",http://10.97.139.98:1454/CAAOI202/CA007001/JF5...,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2538,2025-11-16 11:00:00,aoi100,CAPIC300,B140UAN04,CF,842,OP_Defect,26,857,26,370,161,199,10,0,"AE1U4B05SV,AE1U4B05SW,AE1U4B05SZ,AE1U4B0661,AE...",http://10.97.139.98:1454//CAPIT203/CA0368/AE1U...,None
2539,2025-11-16 11:00:00,aoi100,CAPIC300,B140UAN04,CF,842,PI_Gel,26,857,23,72,33,32,7,0,"AE1U4B05SV,AE1U4B05SW,AE1U4B05SZ,AE1U4B0661,AE...",http://10.97.139.98:1454//CAPIT203/CA0368/AE1U...,None
2540,2025-11-16 11:00:00,aoi100,CAPIC300,B140UAN04,CF,842,PI_Spot_NP,26,857,18,49,9,30,7,3,"AE1U4B05SV,AE1U4B05SZ,AE1U4B0661,AE1U4B0662,AE...",http://10.97.139.98:1454//CAPIT203/CA0368/AE1U...,None
2541,2025-11-16 11:00:00,aoi100,CAPIC300,B140UAN04,CF,842,Polymer,26,857,14,21,1,20,0,0,"AE1U4B05SZ,AE1U4B0662,AE1U4B0663,AE1U4B0665,AE...",http://10.97.139.98:1454//CAPIT203/CA0368/AE1U...,None


In [45]:
main_group = ['scan_time',
 'pi_time',
 'pi_hour',
 'line_id',
 'model',
 'glass_type',
 'recipe_id']

In [19]:
#filter_df = df[(df['glass_id'].str.contains('AR0Q7A0ABH')) & (df['ai_code_1'] =='BM_Cover')]
# df[(df['model'].str.contains('M270QAN07')) ]
filter_df = df[(df['ai_code_1'] =='BM_Cover')][[cl for cl in df.columns if "pic_path" not in cl   ]]
for _, row in filter_df.iterrows():
    if row['maingroup_glass_count'] != row['defect_code_glass_count']:
        print(row)

In [38]:
key_dict = {'pi_hour': '2025-11-15 11', 'line_id': 'CAPIC200', 'aoi': 'aoi100', 'model': 'G215HVN01', 'glass_type': 'CF', 'recipe_id': '608'}
#df["pi_hour"] = pd.to_datetime(df["pi_hour"], errors='coerce')
filter_df = df.copy()
#filter_df['pi_hour'] = filter_df['pi_hour'].apply(lambda x: f'25{str(x)}:00:00')
for key, val in key_dict.items():
    print(key, val)
    if filter_df[filter_df[key] == val].empty:
        break
    else:
        filter_df = filter_df[filter_df[key] == val]
        print('row: ', len(filter_df))
        print(filter_df)



pi_hour 2025-11-15 11
row:  10
                 pi_hour     aoi   line_id      model glass_type recipe_id  \
2479 2025-11-15 11:00:00  aoi100  CAPIC200  G215HVN01         CF       608   
2480 2025-11-15 11:00:00  aoi100  CAPIC200  G215HVN01         CF       608   
2481 2025-11-15 11:00:00  aoi100  CAPIC200  G215HVN01         CF       608   
2482 2025-11-15 11:00:00  aoi100  CAPIC200  G215HVN01         CF       608   
2483 2025-11-15 11:00:00  aoi100  CAPIC200  G215HVN01         CF       608   
2484 2025-11-15 11:00:00  aoi100  CAPIC200  G215HVN01         CF       608   
2485 2025-11-15 11:00:00  aoi100  CAPIC200  G215HVN01         CF       608   
2486 2025-11-15 11:00:00  aoi100  CAPIC200  G215HVN01         CF       608   
2487 2025-11-15 11:00:00  aoi100  CAPIC200  G215HVN01         CF       608   
2488 2025-11-15 11:00:00  aoi100  CAPIC200  G215HVN01         CF       608   

              ai_code_1  maingroup_glass_count  maingroup_defect_count  \
2479           BM_Cover             

In [42]:
filter_df.iloc[-2,:].to_dict()

{'pi_hour': Timestamp('2025-11-15 11:00:00'),
 'aoi': 'aoi100',
 'line_id': 'CAPIC200',
 'model': 'G215HVN01',
 'glass_type': 'CF',
 'recipe_id': '608',
 'ai_code_1': 'Polymer',
 'maingroup_glass_count': 28,
 'maingroup_defect_count': 1506,
 'defect_code_glass_count': 19,
 'defect_code_count': 44,
 'small_defect_count': 16,
 'middle_defect_count': 28,
 'large_defect_count': 0,
 'over_defect_count': 0,
 'glass': 'AL5H1B1MX9,AL5H1B1MXD,AL5H1B1MXF,AL5H1B1MXV,AL5H1B1MY7,AL5H1B1MZ2,AL5H1B1MZC,AL5H1B1MZJ,AL5H1B1N02,AL5H1B1N0B,AL5H1B1N0Z,AL5H1B1N14,AL5H1B1N1Q,AL5H1B1N1W,AL5H1B1N2C,AL5H1B1N2E,AL5H1B1N2K,AL5H1B1N2R,AL5H1B1N36',
 'pic_paths': 'http://10.97.139.98:1454//CAPIT203/CA0367/AL5H1B1MX9/A/20251115152208/,http://10.97.139.98:1454//CAPIT203/CA0367/AL5H1B1MXD/A/20251115145224/,http://10.97.139.98:1454//CAPIT203/CA0367/AL5H1B1MXF/A/20251115140549/,http://10.97.139.98:1454//CAPIT203/CA0367/AL5H1B1MXV/A/20251115140911/,http://10.97.139.98:1454//CAPIT203/CA0367/AL5H1B1MY7/A/20251115142214/,htt

In [35]:
gls = filter_df['glass'].unique()[0].split(",")
len(gls)

19

In [62]:
main_group = ['glass_id','defect_count', 'defect_size', 'ai_code_1',  'pi_time', 'pi_hour']
for main_keys , group_rows in filter_df.groupby(main_group):
    print('分群:', main_keys[1:-1], '筆數', len(group_rows))
    group_rows = group_rows[main_group]
    for _, row in group_rows.iterrows():
        print(row.to_dict())
    #print(group_rows[group_rows['ai_code_1'].isin(cfg.all_defect_codes)][row_keys]) #

分群: ('31', '12', 'NPI_CF', '2025-11-07 08:14:59') 筆數 2
{'glass_id': 'AR0Q7A0ABH', 'defect_count': '31', 'defect_size': '12', 'ai_code_1': 'NPI_CF', 'pi_time': '2025-11-07 08:14:59', 'pi_hour': '2025-11-08 00'}
{'glass_id': 'AR0Q7A0ABH', 'defect_count': '31', 'defect_size': '12', 'ai_code_1': 'NPI_CF', 'pi_time': '2025-11-07 08:14:59', 'pi_hour': '2025-11-08 00'}
分群: ('31', '99', 'NPI_CF', '2025-11-07 08:14:59') 筆數 1
{'glass_id': 'AR0Q7A0ABH', 'defect_count': '31', 'defect_size': '99', 'ai_code_1': 'NPI_CF', 'pi_time': '2025-11-07 08:14:59', 'pi_hour': '2025-11-08 00'}


In [19]:
cols = [ 'glass_id', 'defect_count', 'defect_size', 'ai_code_1','pi_time']

In [51]:
        
"""
for tbn in [tbn for tbn in tbns if '_pidensity_202511_' in tbn]:
    tb = dbhandler.get_table(tbn)
    if tb.empty:continue
    print(f'資料表: {tbn} ,筆數: {len(tb)}')
    tb.fillna('', inplace = True)
    df = tb[tb['pic_path'].str.contains('http://10.97.139.98:1454/CA')]
    for pic_path, pic_rows in df.groupby(['pic_path']):
        pic_path = pic_path[0]
        print(f'影像連結: {pic_path}, 筆數: {len(pic_rows)}')
        print('- ', pic_rows[cols].iloc[0,:].to_dict())

for tbn in [tbn for tbn in tbns if '_pidensity_202511_' in tbn]:
    tb = dbhandler.get_table(tbn)
    if tb.empty:continue
    print(f'資料表: {tbn} ,筆數: {len(tb)}')
    tb.fillna('', inplace = True)
    tb[tb['aoi'] =='aoi100']
    df = tb[tb['pic_path'].str.contains('http://10.97.139.98:1454/CA')]
    for pic_path, pic_rows in df.groupby(['pic_path']):
        pic_path = pic_path[0]
        print(f'影像連結: {pic_path}, 筆數: {len(pic_rows)}')
        print('- ', pic_rows[cols].iloc[0,:].to_dict())
"""
PRIMARY_MAIN_KEYS = ["pi_hour", "line_id", "model", "glass_type", "recipe_id"]
"""
for tbn in [tbn for tbn in tbns if '_pidensity_202511_' in tbn]:
    if tbn != f'aoi200_pidensity_202511_pi200':continue
    tb = dbhandler.get_table(tbn)
    if tb.empty:continue
    print(f'資料表: {tbn} ,筆數: {len(tb)}')
    tb = tb.dropna(subset=['pi_time']).reset_index(drop = True)
    tb['pi_hour'] = tb['pi_time'].apply(lambda x: str(x).split(":")[0])
    min_t, max_t = min(tb['pi_hour']),  max(tb['pi_hour'])
    print(f'dropna 筆數: {len(tb)}, {min_t}, {max_t} ')
    
    #print(tb.head(3))
    code_tb = tb[(tb['model'] =='M215HAN01') & (tb['recipe_id'] =='2231')] #, 'SSIU_Polymer' #(tb['ai_code_1'] .isin(['Polymer'])) & 
    # if bm_tb.empty:continue
    #bm_data = pd.concat([bm_data, bm_tb])
    print(f'code - 筆數: {len(code_tb)}\n---------------------------------')
    for main_key, main_rows in code_tb.groupby(cfg.PRIMARY_GROUP_COLS):
        print(main_key, len(main_rows))
        print('片數:', len(main_rows.groupby(['glass_id', 'pi_hour'])))
        for code, code_rows in main_rows.groupby(['ai_code_1']):
            print(code, len(code_rows))
"""


"""

for tbn in [tbn for tbn in tbns if '_pidensity_202511_' in tbn]:
    #if 'pi100' not in tbn :continue
    if tbn != 'aoi100_pidensity_202511_pi200':continue
    tb = dbhandler.get_table(tbn)
    #
    print(f'資料表: {tbn} ,筆數: {len(tb)}')
    if tb.empty:continue
    tb = tb.dropna(subset=['pi_time']).reset_index(drop = True)
    tb['pi_hour'] = tb['pi_time'].apply(lambda x: str(x).split(":")[0])
    try:
        min_t, max_t = min(tb['pi_hour']),  max(tb['pi_hour'])
        print(f'--- dropna 筆數: {len(tb)}, pi_time時間: {min_t} ~  {max_t} ')
    except:
        uni_times = tb['pi_hour'].unique()
        print(f'--- dropna 筆數: {len(tb)}, pi_time時間: {uni_times}' )
    #
    print("分群:", len(tb.groupby(cfg.PRIMARY_GROUP_COLS)))
    for main_key, main_rows in  tb.groupby(cfg.PRIMARY_GROUP_COLS):
        print(main_key, len(main_rows))
        #print(main_rows[cols])
    
    #print(tb.head(3))
    code_tb = tb[(tb['model'] =='M215HAN01') & (tb['recipe_id'] =='2231')] #, 'SSIU_Polymer' #(tb['ai_code_1'] .isin(['Polymer'])) & 
    # if bm_tb.empty:continue
    #bm_data = pd.concat([bm_data, bm_tb])
    print(f'code - 筆數: {len(code_tb)}\n---------------------------------')
    for main_key, main_rows in code_tb.groupby(cfg.PRIMARY_GROUP_COLS):
        print(main_key, len(main_rows))
        print('片數:', len(main_rows.groupby(['glass_id', 'pi_hour'])))
        for code, code_rows in main_rows.groupby(['ai_code_1']):
            print(code, len(code_rows))
#
"""



for tbn in [tbn for tbn in tbns if '_pidensity_202511_' in tbn]:
    #if 'pi100' not in tbn :continue
    if tbn != 'aoi100_pidensity_202511_pi200':continue
    tb = dbhandler.get_table(tbn)
    
    print(f'資料表: {tbn} ,筆數: {len(tb)}')
    if tb.empty:continue
    tb = tb.dropna(subset=['pi_time']).reset_index(drop = True)
    tb['pi_hour'] = tb['pi_time'].apply(lambda x: str(x).split(":")[0])
    tb = tb[tb['pi_hour'] == '2025-11-15 11']

資料表: aoi100_pidensity_202511_pi200 ,筆數: 5970


In [54]:
for main_key, main_rows in tb.groupby(cfg.PRIMARY_GROUP_COLS):
    print(main_key, len(main_rows))
    #for defect, defect_rows in main_rows.groupby(['ai_code_1', 'glass_id']):
    #    print(defect, )
    #main_rows = main_rows[main_rows['ai_code_1'] == 'SSIU_Polymer']
    glass_groups = len(main_rows.groupby(['glass_id']))
    print(f'glass_num: {glass_groups}')
    #for glass, glass_rows in main_rows.groupby(['glass_id']):
    #    print(glass, len(glass_rows))
    main_rows.to_csv('aoi100_pidensity_202511_pi200_2025111511_CAPIC200_G215HVN01_CF_608.csv')
    

('2025-11-15 11', 'CAPIC200', 'G215HVN01', 'CF', '608') 1506
glass_num: 28


In [ ]:
num1 = 0
for id, glass_rows in main_rows.groupby(['glass_id']):
    print(id, len(glass_rows))
    num1 +=1
num1

('AL5H1B1MX9',) 2
('AL5H1B1MXD',) 5
('AL5H1B1MXF',) 3
('AL5H1B1MXG',) 5
('AL5H1B1MXN',) 4
('AL5H1B1MXR',) 5
('AL5H1B1MXV',) 5
('AL5H1B1MY7',) 4
('AL5H1B1MZ2',) 8
('AL5H1B1MZC',) 9
('AL5H1B1MZJ',) 4
('AL5H1B1N02',) 5
('AL5H1B1N05',) 6
('AL5H1B1N0B',) 2
('AL5H1B1N0Z',) 3
('AL5H1B1N14',) 4
('AL5H1B1N1J',) 4
('AL5H1B1N1Q',) 1
('AL5H1B1N1W',) 2
('AL5H1B1N2C',) 2
('AL5H1B1N2E',) 1
('AL5H1B1N2J',) 6
('AL5H1B1N2K',) 3
('AL5H1B1N2R',) 4
('AL5H1B1N36',) 6
('AL5H1B1N3D',) 1
('AL5H1B1N3X',) 8


27

In [23]:
tb = dbhandler.get_table("pidensity_202511")
tb[tb['aoi']=='aoi100']

,pi_hour,aoi,pi,line_id,model,glass_type,recipe_id,defect_type,ai_code_1,n_rows,n_glasses,small_defect_count,middle_defect_count,large_defect_count,over_defect_count,unknown_defect_count,glass,noteText


In [11]:
#Raw_dict = {}
cl = 'pi_time'
for tbn in [tbn for tbn in tbns if '_pidensity_202511_' in tbn]:
    tb = dbhandler.get_table(tbn)
    if tb.empty:continue
    
    tb[cl] = pd.to_datetime(tb[cl], errors="coerce")
    tb = tb.sort_values([cl], ascending=False)
    tb.reset_index(inplace = True , drop = True)
    latest_time = max(tb[cl])
    uni_times = tb[cl].unique().tolist()
    #print(f'*** 資料表: {tbn} ****\n筆數: {len(tb)}, 最新時間:{latest_time}')# (時間分群: {uni_times})')
    print(f'*** 資料表: {tbn} **** , 最新時間:{latest_time}')# (時間分群: {uni_times})')
    tb = tb[['pi_time', 'pi_hour', 'scan_time', 'line_id', 'model', 'glass_type', 'recipe_id', 'glass_id','pic_name', 'defect_count', 'defect_size', 'ai_code_1']]
    uni_times = tb[cl].unique().tolist()
    """
    try:
        print('-- 最新一筆資料:\n',tb[tb['pi_time'] == latest_time].iloc[0,:].to_dict(), f'-- 時間分群:{uni_times}')
        #print(tb.loc[[tb.index[0],  tb.index[-1]], tb.columns].to_dict())
        
    except:
        print(f'-- 時間資料錯誤:{uni_times}')
    """
    #print("----------")
    


*** 資料表: aoi100_pidensity_202511_pi300 **** , 最新時間:2025-11-07 23:53:31
*** 資料表: aoi100_pidensity_202511_pi400 **** , 最新時間:NaT
*** 資料表: aoi100_pidensity_202511_pi500 **** , 最新時間:2025-11-03 19:50:09
*** 資料表: aoi100_pidensity_202511_pi700 **** , 最新時間:2025-11-08 00:22:50
*** 資料表: aoi200_pidensity_202511_pi100 **** , 最新時間:2025-11-07 11:23:27
*** 資料表: aoi200_pidensity_202511_pi200 **** , 最新時間:2025-11-07 06:49:57
*** 資料表: aoi200_pidensity_202511_pi300 **** , 最新時間:2025-11-07 07:20:41
*** 資料表: aoi200_pidensity_202511_pi400 **** , 最新時間:2025-11-05 23:25:07
*** 資料表: aoi200_pidensity_202511_pi500 **** , 最新時間:2025-11-07 06:44:11
*** 資料表: aoi200_pidensity_202511_pi600 **** , 最新時間:2025-11-05 23:58:10
*** 資料表: aoi200_pidensity_202511_pi700 **** , 最新時間:2025-11-05 22:28:28


In [15]:
a = 3029-10827
b = 85016-84415
c = 2740-2447
d = 33781-31871
e= 281-21163
ts = [a, b, c, d, e ]
print(ts)
print(sum(ts))

50000+sum(ts)

[-7798, 601, 293, 1910, -20882]
-25876


24124

In [ ]:
key_dict = {
 'aoi': 'AOI100',
 'pi': 'PI700',
 'line_id': 'CAPIC700',
 'model': 'M270HVN0G',
 'glass_type': 'CF',
 'recipe_id': '611',
 'defect_type': '',
 'ai_code_1': 'BM_Cover',
 'n_rows': 63,
 'n_glasses': 8,
 'small_defect_count': 62,
 'middle_defect_count': 1,
 'large_defect_count': 0,
 'over_defect_count': 0,
 'unknown_defect_count': 0,
 'glass': 'AR0HG812TJ,AR0HG813T2,AR0HG813WA,AR0HG814UZ,AR0HG8154L,AR0HG8155W,AR0HG8159S,AR0HG8159X',
 }

update_dict = {'noteText':'test'}

In [6]:
tbn = 'aoi200_pidensity_202510_pi200'
#tbn = 'pidensity_202510'
#tbn = 'spec_pro_update_history'
#tbn = 'spec_pro_update__table'
tb2 = dbhandler.get_table(tbn)
tb = dbhandler.get_table(tbn)
#set([str(v).split(" ")[0] for v in tb['pi_hour'].unique()])
tb.iloc[0,:].to_dict()

{'scan_time': Timestamp('2025-10-01 06:07:53'),
 'line_id': 'CAPIC200',
 'model': 'M270DANA0',
 'glass_type': 'TFT',
 'recipe_id': '2236',
 'glass_id': 'VN5A9N49J',
 'pic_name': 'CaptureImage/RV1_289334_1304326_0',
 'x': '289334',
 'y': '1304326',
 'predict_code': '1',
 'judge_code': '1',
 'mark': '',
 'hour': '',
 'dayhour': '',
 'day': Timestamp('2025-10-01 06:07:53'),
 'year': '',
 'month': '',
 'season': '',
 'week': '',
 'yearmonth': '',
 'defect_count': '36',
 'defect_size': '8',
 'open_status': '',
 'ai_code_1': 'OP_Defect',
 'ai_code_2': '2025-10-01 04:15:05',
 'ai_code_3': '',
 'ai_code_4': '',
 'ai_code_5': '',
 'gray_name': 'CCDImage\\IP01\\01_04_0018',
 'ip_num': '',
 'first_code': '',
 'chip_name': 'A',
 'defect_seq': '1'}

In [ ]:
{'scan_time': '',
 'line_id': 'CAPIC200',
 'model': 'M270DANA0',
 'glass_type': 'TFT',
 'glass_id': 'VN5A9N49J',
 'defect_size': '8',
 'ai_code_1': 'OP_Defect',
 }.keys()

In [ ]:
summart_tb = dbhandler.get_table('pidensity_202510')
summart_tb

## map defect重疊趨勢
## density by day, mon, ..趨勢

動態density spec計算

In [ ]:
{'line_id': [], 
 'model': [], 
 'glass_type':[], 
 'ai_code_1': []}.keys()


In [ ]:
cfg.defect_group_coldict

In [4]:
from itertools import combinations
import pandas as pd


def split_into_15_size_rows(df):
    """
    將 df 依 {S,M,L,O} 所有非空子集合切成 15 組，彼此會重疊。
    需要欄位: 'size_bucket'
    """
    assert 'size_bucket' in df.columns, "df 需包含欄位 'size_bucket'"
    ORDER = ['S', 'M', 'L', 'O']
    out = {}
    for r in range(1, len(ORDER) + 1):
        for combo in combinations(ORDER, r):
            key = ''.join(combo)
            sub = df.loc[df['size_bucket'].isin(combo)].copy()
            out[key] = sub
    return out


def split_by_glass_type(df, glass_col='glass_type', types=('CF', 'TFT')):
    """
    回傳 {'ALL': 全部, 'CF': ..., 'TFT': ...}
    空的 CF/TFT 就不要放進去。
    """
    out = {'ALL': df.copy()}
    for t in types:
        sub = df.loc[df[glass_col] == t].copy()
        
        if not sub.empty:
            out[t] = sub
    return out

#def 
def build_groups_mode1(tb, size_group_keys):
    total_group_dict = {}

    for base_keys, base_df in tb.groupby(['line_id', 'aoi', 'model', 'recipe_id', 'ai_code_1'], dropna=False):
        # base_keys = (line_id, aoi, model, recipe_id, ai_code_1)
        base_df = base_df[size_group_keys]

        # glass_type 要有 ALL
        glass_dict = split_by_glass_type(base_df, glass_col='glass_type', types=('CF', 'TFT'))

        for glass_type, gdf in glass_dict.items():
            size_dict = split_into_15_size_rows(gdf)

            for size_key, sdf in size_dict.items():
                if sdf.empty:
                    continue

                # key: line_id:aoi:model:recipe_id:ai_code_1:glass_type:size_key
                parts = [*base_keys, glass_type, size_key]
                keyword = ":".join(str(v) for v in parts)

                if keyword in total_group_dict:
                    total_group_dict[keyword] = pd.concat(
                        [total_group_dict[keyword], sdf],
                        ignore_index=True
                    )
                else:
                    total_group_dict[keyword] = sdf.reset_index(drop=True)

    return total_group_dict


def build_groups_mode2(tb, size_group_keys):
    total_group_dict = {}

    for base_keys, base_df in tb.groupby(['line_id', 'aoi', 'model', 'recipe_id'], dropna=False):
        base_df = base_df[size_group_keys]
        print(base_keys)
        # glass_type 要有 ALL
        glass_dict = split_by_glass_type(base_df, glass_col='glass_type', types=('CF', 'TFT'))
        
        
        for glass_type, gdf in glass_dict.items():
            print(f'→ glass_type: {glass_type}, {len(gdf)}')
            # 在這個 glass_type 底下，再按實際出現的 ai_code_1 切
            for ai_code, aidf in gdf.groupby('ai_code_1', dropna=False):
                ai_code_str = str(ai_code) if pd.notna(ai_code) else 'NA'
                print(f'→→ ai_code: {ai_code_str}, {len(aidf)}')

                size_dict = split_into_15_size_rows(aidf)

                for size_key, sdf in size_dict.items():
                    if sdf.empty:
                        continue

                    # key: line_id:aoi:model:recipe_id:glass_type:ai_code_1:size_key
                    parts = [*base_keys, glass_type, ai_code_str, size_key]
                    keyword = ":".join(str(v) for v in parts)

                    if keyword in total_group_dict:
                        total_group_dict[keyword] = pd.concat(
                            [total_group_dict[keyword], sdf],
                            ignore_index=True
                        )
                    else:
                        total_group_dict[keyword] = sdf.reset_index(drop=True)
        

    return total_group_dict



def build_groups(tb, size_group_keys, mode='glass_first'):
    if mode == 'ai_first':
        return build_groups_mode1(tb, size_group_keys)
    elif mode == 'glass_first':
        return build_groups_mode2(tb, size_group_keys)
    else:
        raise ValueError("mode 必須是 'ai_first' 或 'glass_first'")


In [6]:

def filter_by_first_or_len3(df, col, first_char):
    """
    保留 df[col] 第一個字元等於 first_char，或字串長度等於 3 的列。
    - 自動把資料轉成字串並 strip 空白
    - 對 NaN 友善（當成不符合）
    """
    s = df[col].astype('string').str.strip()        # 統一為字串並去頭尾空白
    mask = s.str.startswith(str(first_char), na=False) | (s.str.len() == 3)
    return df[mask]

def group_clean_same_glass_diif_scantime(gdf):
    drop_idxs = []
    print(f'total rows: {len(gdf)}')
    for glass, grows in gdf.groupby(['line_id', 'aoi','model', 'recipe_id', 'glass_type', 'glass_id']):
        
        
        if len(grows['scan_time'].unique()) !=1:
            uni_times = grows['scan_time'].unique().tolist()
           
            idxs = grows[grows['scan_time'].isin( grows['scan_time'].unique()[:-1]) ].index.tolist()
            drop_idxs = drop_idxs + idxs
            print(glass, len(grows),uni_times, f'→ del:{len(idxs)}')
    
    if len(drop_idxs) !=0:
        gdf.drop(drop_idxs, axis = 0, inplace = True)
        gdf.reset_index(inplace = True, drop = True)
    print(f'刪除同片重複量測資料後: total rows → {len(gdf)}')
    
    return gdf

In [ ]:
['scan_time', 'line_id', 'model', 'recipe_id', 'glass_type','glass_id', 'ai_code_1', 'defect_size']

In [9]:
rows = {}
def_codes = cfg.uni_UPI_defect_codes + cfg.uni_SPOT_defect_codes
_now_tpe = pd.Timestamp.now(tz='Asia/Taipei')
_cutoff = (_now_tpe - pd.Timedelta(days=90)).tz_localize(None)

mode1_size_group_keys = ['scan_time','glass_type','glass_id','size_bucket']
mode2_size_group_keys = ['scan_time','glass_type','glass_id','ai_code_1','size_bucket'] #


size_key_names = ['S','M','L','O','SM','SL','SO','ML','MO','LO','SML','SMO','SLO','MLO','SMLO']
spec_data_keys = ['size_group', 'glass_num', 'defect_num', 'density', 'std', 'ooc', 'oos']
data_keys = ['scan_time', 'line_id', 'model', 'recipe_id', 'glass_type','glass_id', 'ai_code_1', 'defect_size']

total_group_dict = {}
total_group_df = pd.DataFrame()

for tbn in [tbn for tbn in tbns if '_pidensity_' in tbn]:
    tb = dbhandler.get_table(tbn)
    tb = tb[data_keys].copy()
    if tb.empty :continue
    print(f'#########  {tbn} ###########')

    # 1. --- 建立 pi_hour 與 90 天內過濾 ---
    # 轉成 datetime
    tb['scan_time'] = pd.to_datetime(tb['scan_time'], errors='coerce')
    """
    tb = tb.loc[tb['scan_time'].notna()].copy()
    tb = tb.loc[tb['scan_time'] >= _cutoff].copy()
    """
    if tb.empty :continue

    # 2. 只取關注的 defect codes
    #tb = tb[tb['ai_code_1'].isin(def_codes)].copy()
    
    filter_df = pd.DataFrame()
    for char in ['2','0']:
        filter_rows = filter_by_first_or_len3(tb, 'recipe_id', char)
        print(char, len(filter_rows), filter_rows['recipe_id'].unique())
        if filter_rows.empty:continue
        filter_df = pd.concat([ filter_df ,filter_rows])
    
    if filter_df.empty :continue
        
    else:
        tb = filter_df.copy()
    
    """
    #用pi_hour去做篩選>= _cutoff
    # 產生 pi_hour（字串），也保留小時對齊的 datetime（之後做 groupby 好用）
    tb['pi_hour'] = tb['scan_time'].dt.strftime('%Y-%m-%d %H')
    tb['pi_hour_dt'] = tb['scan_time'].dt.floor('h')
    _cutoff_hour = pd.Timestamp(_cutoff).floor('h')
    tb = tb.loc[tb['pi_hour_dt'] >= _cutoff_hour].copy()
    """
    
    # 3. --- 依規則分群 defect_size 到 S/M/L/O ---
    # 先確保 defect_size 是數值
    #"""
    tb['defect_size'] = pd.to_numeric(tb.get('defect_size'), errors='coerce')

    # 以 bins 對應你的規則：
    # S: <=20, M: 21-100, L: 101-400, O: >=401
    bins = [-float('inf'), 20, 100, 400, float('inf')]
    labels = ['S', 'M', 'L', 'O']
    tb['size_bucket'] = pd.cut(tb['defect_size'], bins=bins, labels=labels, right=True)

    tb = tb.loc[tb['size_bucket'].notna()].copy()
    #"""
    if len(tb) == 0:continue


    tb['aoi'] = tbn.split("_")[0]
    total_group_df = pd.concat([total_group_df, tb])

total_group_df.reset_index(inplace = True, drop = True)
total_group_df = group_clean_same_glass_diif_scantime(total_group_df)

#break

#########  aoi100_pidensity_202509_pi100 ###########
2 7963 ['815']
0 7963 ['815']
#########  aoi100_pidensity_202509_pi200 ###########
2 7983 ['117' '602' '608']
0 7983 ['117' '602' '608']
#########  aoi100_pidensity_202509_pi300 ###########
2 3685 ['105' '842']
0 3685 ['105' '842']
#########  aoi100_pidensity_202509_pi500 ###########
2 10484 ['822' '611']
0 10484 ['822' '611']
#########  aoi100_pidensity_202509_pi700 ###########
2 13740 ['611' '805' '205' '605' '806']
0 13740 ['611' '805' '205' '605' '806']
#########  aoi100_pidensity_202510_pi100 ###########
2 4036 ['117' '205' '105']
0 4036 ['117' '205' '105']
#########  aoi100_pidensity_202510_pi200 ###########
2 33735 ['608' '842' '803' '814']
0 33735 ['608' '842' '803' '814']
#########  aoi100_pidensity_202510_pi300 ###########
2 9597 ['842' '202' '205']
0 9597 ['842' '202' '205']
#########  aoi100_pidensity_202510_pi400 ###########
2 2516 ['842']
0 2516 ['842']
#########  aoi100_pidensity_202510_pi500 ###########
2 676 ['205']


In [30]:
total_group_df.head(3)

,scan_time,line_id,model,recipe_id,glass_type,glass_id,ai_code_1,defect_size,size_bucket,aoi
0,2025-09-16 15:48:54,CAPIC100,B116XAN06,815,CF,AA1X6900TD,OP_Defect,68,M,aoi100
1,2025-09-16 15:48:54,CAPIC100,B116XAN06,815,CF,AA1X6900TD,SSIU_Polymer,12,S,aoi100
2,2025-09-16 15:48:54,CAPIC100,B116XAN06,815,CF,AA1X6900TD,OP_Defect,30,M,aoi100


In [32]:
def count_group_statis(df, total_glass_count):
    df.reset_index(inplace=True, drop = True)
    glass_groups = df.groupby(['scan_time', 'glass_id'])
    uni_size_glass_count = len(glass_groups)
    uni_size_defect_count = len(df)
   
    size_density = uni_size_defect_count / total_glass_count
    over_ave = size_density * 3
    print(f'→→→ size: {size_key}, defect: {uni_size_defect_count}, glass count:{uni_size_glass_count}\nover_ave: {over_ave}')
    return df, glass_groups, uni_size_defect_count, size_density, over_ave

total_group_dict = {}
size_group_keys = ['scan_time','glass_type','glass_id','ai_code_1','size_bucket'] 
ai_codes = cfg.uni_UPI_defect_codes + cfg.uni_SPOT_defect_codes + cfg.uni_SPS_defect_codes
for base_keys, base_df in total_group_df.groupby(['line_id', 'aoi', 'model', 'recipe_id'], dropna=False):
    base_df = base_df[size_group_keys]
    print(base_keys)
    # glass_type 要有 ALL
    glass_dict = split_by_glass_type(base_df, glass_col='glass_type', types=('CF', 'TFT'))

    for glass_type, gdf in glass_dict.items():
        total_glass_count = len(gdf.groupby(['scan_time', 'glass_id']))
        total_defect_count = len(gdf)
        print(f'→ glass_type: {glass_type}, total defect: {total_defect_count}, glass count: {total_glass_count}')

        # 在這個 glass_type 底下，再按實際出現的 ai_code_1 切
        for ai_code, aidf in gdf.groupby('ai_code_1', dropna=False):
            ai_code_str = str(ai_code) if pd.notna(ai_code) else 'NA'
            if ai_code not in ai_codes:continue
            uni_code_glass_count = len(aidf.groupby(['scan_time', 'glass_id']))
            print(f'→→ ai_code: {ai_code_str},  defect: {len(aidf)}, glass count:{uni_code_glass_count}')

            size_dict = split_into_15_size_rows(aidf)

            for size_key, sdf in size_dict.items():
                if sdf.empty:
                    continue
                sdf, glass_group_data, uni_size_defect_count, size_density, over_ave = count_group_statis(sdf, total_glass_count)
                over_data = [glass_rows.index for glass_keys, glass_rows in glass_group_data if len(glass_rows) >= over_ave]
                
                if over_data:
                    print(f'超過{over_ave} → 刪除去除離群glass片數: {len(over_data)}')
                    del_glass_num = 0
                    for idxs in over_data:
                        sdf.drop(idxs, axis = 0, inplace=True)
                        del_glass_num+=1
                    if not sdf.empty:
                        glass_num =  total_glass_count -del_glass_num
                        sdf, glass_group_data, uni_size_defect_count, size_density, _ = count_group_statis(sdf, glass_num)
                else:
                    glass_num =  total_glass_count
                print('size_density',size_density)
                subs = [(len(glass_rows)- size_density)**2 for _, glass_rows in glass_group_data ]
                print("subs",subs)
                if uni_size_defect_count <= 1 or not isinstance(glass_num, (int, float)) or len(subs) ==0 :
                    print(f"數據量不足")
                    std = ooc = oos = np.nan
                else:
                    
                    std = (sum(subs) / (glass_num - 1)) ** 0.5 
                    ooc = round( size_density + 3 * std, 2) 
                    oos = round( size_density + 6 * std, 2) 

                spec_vals = [round(v, 2) if  isinstance(v, (int, float))  else np.nan for v in [glass_num, uni_size_defect_count, size_density, over_ave, std, ooc, oos] ]
                print(spec_vals)
                # key: line_id:aoi:model:recipe_id:glass_type:ai_code_1:size_key
                
                """
                parts = [*base_keys, glass_type, ai_code_str, size_key]
                keyword = ":".join(str(v) for v in parts)

                if keyword in total_group_dict:
                    total_group_dict[keyword] = pd.concat(
                        [total_group_dict[keyword], sdf],
                        ignore_index=True
                    )
                else:
                    total_group_dict[keyword] = sdf.reset_index(drop=True)
                """
        break
    break


('CAPIC100', 'aoi100', 'B116XAN06', '815')
→ glass_type: ALL, total defect: 15926, glass count: 82
→→ ai_code: PIS With Particle,  defect: 442, glass count:60
→→→ size: S, defect: 114, glass count:28
over_ave: 4.170731707317072
超過4.170731707317072 → 刪除去除離群glass片數: 7
→→→ size: S, defect: 56, glass count:21
over_ave: 2.24
size_density 0.7466666666666667
subs [1.5708444444444447, 10.584177777777779, 1.5708444444444447, 1.5708444444444447, 10.584177777777779, 10.584177777777779, 10.584177777777779, 1.5708444444444447, 10.584177777777779, 1.5708444444444447, 10.584177777777779, 1.5708444444444447, 1.5708444444444447, 1.5708444444444447, 1.5708444444444447, 1.5708444444444447, 10.584177777777779, 1.5708444444444447, 1.5708444444444447, 1.5708444444444447, 1.5708444444444447]
[75, 56, 0.75, 4.17, 1.14, 4.17, 7.58]
→→→ size: M, defect: 276, glass count:55
over_ave: 10.097560975609756
超過10.097560975609756 → 刪除去除離群glass片數: 4
→→→ size: M, defect: 226, glass count:51
over_ave: 8.692307692307692
si

IndexError: indices are out-of-bounds

In [26]:
# === 主要分組 'ai_first' 或 'glass_first' ===
#group_dict = build_groups(tb,mode2_size_group_keys , mode='glass_first')
group_dict = build_groups(total_group_df,mode2_size_group_keys , mode='glass_first')
#break
for k, v in group_dict.items():
    if v.empty:
        continue
    if k in total_group_dict:
        total_group_dict[k] = pd.concat(
            [total_group_dict[k], v],
            ignore_index=True
        )
    else:
        total_group_dict[k] = v.reset_index(drop=True)

('CAPIC100', 'aoi100', 'B116XAN06', '815')
→ glass_type: ALL, 7963
→→ ai_code: BM_Cover, 11
→→ ai_code: NPI_OIL, 303
→→ ai_code: NPI_TFT, 526
→→ ai_code: Not_Found, 306
→→ ai_code: OP_Defect, 3251
→→ ai_code: PIS With Particle, 221
→→ ai_code: PI_Gel, 3
→→ ai_code: PI_Spot_NP, 45
→→ ai_code: Polymer, 135
→→ ai_code: SSIU_Polymer, 3162
→ glass_type: CF, 7963
→→ ai_code: BM_Cover, 11
→→ ai_code: NPI_OIL, 303
→→ ai_code: NPI_TFT, 526
→→ ai_code: Not_Found, 306
→→ ai_code: OP_Defect, 3251
→→ ai_code: PIS With Particle, 221
→→ ai_code: PI_Gel, 3
→→ ai_code: PI_Spot_NP, 45
→→ ai_code: Polymer, 135
→→ ai_code: SSIU_Polymer, 3162
('CAPIC100', 'aoi100', 'M215HAN01', '117')
→ glass_type: ALL, 3425
→→ ai_code: Fiber, 4
→→ ai_code: Metal, 30
→→ ai_code: NPI_CF, 810
→→ ai_code: NPI_OIL, 176
→→ ai_code: NPI_TFT, 157
→→ ai_code: Not_Found, 113
→→ ai_code: OP_Defect, 395
→→ ai_code: PI_Gel, 10
→→ ai_code: PI_Spot_Abnormal, 1
→→ ai_code: PI_Spot_NP, 53
→→ ai_code: Polymer, 654
→→ ai_code: SPS, 220
→→ a

In [ ]:
total_group_dict_2 = {}
for group_key, mrows in total_group_dict.items():
    print(group_key, len(mrows))
    group_key2 = ":".join(group_key.split(":")[:-2])
    print(group_key2)
    #if not in total_group_dict_2.keys():
    break   

CAPIC700:aoi200:B156HTN06:0269:ALL:BM_Cover:M 10
['CAPIC700', 'aoi200', 'B156HTN06', '0269', 'ALL']


('AA1X69000C',) 147 <DatetimeArray>
['2025-09-16 16:38:47']
Length: 1, dtype: datetime64[ns]
('AA1X69000D',) 147 <DatetimeArray>
['2025-09-16 16:02:16']
Length: 1, dtype: datetime64[ns]
('AA1X69000R',) 147 <DatetimeArray>
['2025-09-16 15:57:51']
Length: 1, dtype: datetime64[ns]
('AA1X69000T',) 194 <DatetimeArray>
['2025-09-16 17:53:22']
Length: 1, dtype: datetime64[ns]
('AA1X69003D',) 153 <DatetimeArray>
['2025-09-16 17:48:45']
Length: 1, dtype: datetime64[ns]
('AA1X69003E',) 152 <DatetimeArray>
['2025-09-16 17:34:26']
Length: 1, dtype: datetime64[ns]
('AA1X69003K',) 183 <DatetimeArray>
['2025-09-16 17:38:56']
Length: 1, dtype: datetime64[ns]
('AA1X69003M',) 164 <DatetimeArray>
['2025-09-16 17:24:58']
Length: 1, dtype: datetime64[ns]
('AA1X69003P',) 173 <DatetimeArray>
['2025-09-16 17:29:37']
Length: 1, dtype: datetime64[ns]
('AA1X69003Q',) 147 <DatetimeArray>
['2025-09-16 17:15:51']
Length: 1, dtype: datetime64[ns]
('AA1X69003S',) 140 <DatetimeArray>
['2025-09-16 17:20:25']
Length: 1,

In [24]:
def count_group_statis(df, total_glass_count):
    df.reset_index(inplace=True, drop = True)
    glass_groups = df.groupby(['scan_time', 'glass_id'])
    uni_size_glass_count = len(glass_groups)
    uni_size_defect_count = len(df)
    print(f'→→→ size: {size_key}, defect: {uni_size_defect_count}, glass count:{uni_size_glass_count}')
    size_density = uni_size_defect_count / total_glass_count
    over_ave = size_density * 3
    return df, glass_groups, uni_size_defect_count, size_density, over_ave

In [58]:
print(len(total_group_dict.keys()))
#pd.DataFrame(columns = )
total_density_data = []
for group_name, group_rows in total_group_dict.items():
    print(group_name, len(group_rows))
    group_keywords = group_name.split(":")
    if group_rows.empty: continue
    
    
    group_rows.reset_index(drop = True, inplace=True)
    glass_group_data, glass_count, defect_count, group_density, over_ave = count_group_statis(group_rows)

    #print('----', glass_count, defect_count, round(group_density,1),over_std )
    over_data = [ glass_rows.index for glass_names, glass_rows in glass_group_data if len(glass_rows) > over_ave ]
    if len(over_data) !=0:
        print(f'去除離群glass片數: {len(over_data)}')
        for idxs in over_data:
            group_rows.drop([v for v in idxs if v in group_rows.index], axis = 0, inplace=True)
        group_rows.reset_index(drop = True, inplace=True)
        glass_group_data, glass_count, defect_count, group_density, over_ave = count_group_statis(group_rows)
    #print(glass_count, defect_count, group_density, over_ave)
    subs = [(len(glass_rows)- group_density)**2 for _, glass_rows in glass_group_data if len(glass_rows) < over_ave ]
    #print("subs",subs)
    if glass_count <= 1 or not isinstance(glass_count, (int, float)) or len(subs) ==0 :
        print(f"數據量不足")
        std = ooc = oos = np.nan
    else:
        std = (sum(subs) / (glass_count - 1)) ** 0.5 
        ooc = round( group_density + 3 * std, 2) 
        oos = round( group_density + 6 * std, 2) 
        
    spec_vals = [round(v, 2) if  isinstance(v, (int, float))  else np.nan for v in [ glass_count, defect_count, group_density, over_ave, std, ooc, oos] ]
    vals = group_keywords + spec_vals
    total_density_data.append(vals)


    
    

9596
CAPIC100:aoi100:B116XAN06:815:PIS With Particle:ALL:S 57
去除離群glass片數: 1
CAPIC100:aoi100:B116XAN06:815:PIS With Particle:ALL:M 138
CAPIC100:aoi100:B116XAN06:815:PIS With Particle:ALL:L 25
CAPIC100:aoi100:B116XAN06:815:PIS With Particle:ALL:O 1
數據量不足
CAPIC100:aoi100:B116XAN06:815:PIS With Particle:ALL:SM 195
CAPIC100:aoi100:B116XAN06:815:PIS With Particle:ALL:SL 82
去除離群glass片數: 1
CAPIC100:aoi100:B116XAN06:815:PIS With Particle:ALL:SO 58
去除離群glass片數: 1
CAPIC100:aoi100:B116XAN06:815:PIS With Particle:ALL:ML 163
CAPIC100:aoi100:B116XAN06:815:PIS With Particle:ALL:MO 139
CAPIC100:aoi100:B116XAN06:815:PIS With Particle:ALL:LO 26
CAPIC100:aoi100:B116XAN06:815:PIS With Particle:ALL:SML 220
CAPIC100:aoi100:B116XAN06:815:PIS With Particle:ALL:SMO 196
CAPIC100:aoi100:B116XAN06:815:PIS With Particle:ALL:SLO 83
去除離群glass片數: 1
CAPIC100:aoi100:B116XAN06:815:PIS With Particle:ALL:MLO 164
CAPIC100:aoi100:B116XAN06:815:PIS With Particle:ALL:SMLO 221
CAPIC100:aoi100:B116XAN06:815:PIS With Particle:CF

In [53]:
len(total_density_data)

9596

In [59]:

mode1_spec_table_keys = ['line_id','aoi', 'model', 'recipe_id', 'ai_code_1','glass_type', 'defect_size','glass_count',
       'defect_count', 'density','first_over_std', 'std', 'ooc','oos']
mode2_spec_table_keys = ['line_id','aoi', 'model', 'recipe_id','glass_type', 'ai_code_1', 'defect_size','glass_count',
       'defect_count', 'density','first_over_std', 'std', 'ooc','oos']

ai_side_df = pd.DataFrame(total_density_data, columns = mode1_spec_table_keys) #
print(len(ai_side_df))
ai_side_df.head()
dbhandler.save_table('ai_side_spec_table', ai_side_df)
"""
side_ai_df = pd.DataFrame(total_density_data, columns = mode2_spec_table_keys) #
print(len(side_ai_df))
dbhandler.save_table('side_ai_spec_table', side_ai_df)
"""
#side_ai_df.head()


9596


2025-10-30 16:17:46,234 - INFO - [save_table] l6a01_project.ai_side_spec_table 已寫入 9596 列（去除 0 重複）


"\nside_ai_df = pd.DataFrame(total_density_data, columns = mode2_spec_table_keys) #\nprint(len(side_ai_df))\ndbhandler.save_table('side_ai_spec_table', side_ai_df)\n"

In [ ]:

ai_side_table = dbhandler.get_table('ai_side_spec_table')
side_ai_table = dbhandler.get_table('side_defcode_spec_table')
ai_side_table.head()
side_ai_table .head()


In [ ]:
match_num = 0
match_fail = {}
for ix, row in ai_side_table.iterrows():
    test_df = side_ai_table.copy()
    c = 0
    for cl, val in row.to_dict().items():
        #
        if cl in ['glass_count','defect_count']:
            test_df[cl] = pd.to_numeric(test_df[cl], errors = 'coerce')

        test_df2 = test_df[test_df[cl]==val]
        if test_df2.empty:
            c +=1
            #print(test_df)
            match_fail[ix] = {
                'cl1':cl,
                'val1':val,
                'val2':test_df.to_dict()[cl],
                'side_ai': test_df.iloc[0,:].to_dict(),
                'ai_side':row.to_dict()
            }
            print(match_fail[ix])
            break
            #print(f"Match fail({cl}:{val})\n{row.to_dict()}")
        else:
            test_df = test_df2 .copy()

    if c ==0 and not test_df.empty:
        continue
        

{'cl1': 'density', 'val1': 1.81, 'val2': {0: '1.81'}, 'side_ai': {'line_id': 'CAPIC100', 'aoi': 'aoi100', 'model': 'B116XAN06', 'recipe_id': '815', 'ai_code_1': 'PIS With Particle', 'glass_type': 'ALL', 'defect_size': 'S', 'glass_count': 27.0, 'defect_count': 49.0, 'density': '1.81', 'first_over_std': '5.44', 'std': '1.04', 'ooc': '4.93', 'oos': '8.05'}, 'ai_side': {'line_id': 'CAPIC100', 'aoi': 'aoi100', 'model': 'B116XAN06', 'recipe_id': '815', 'ai_code_1': 'PIS With Particle', 'glass_type': 'ALL', 'defect_size': 'S', 'glass_count': 27, 'defect_count': 49, 'density': 1.81, 'first_over_std': 5.44, 'std': 1.04, 'ooc': 4.93, 'oos': 8.05}}
{'cl1': 'density', 'val1': 2.51, 'val2': {1: '2.51'}, 'side_ai': {'line_id': 'CAPIC100', 'aoi': 'aoi100', 'model': 'B116XAN06', 'recipe_id': '815', 'ai_code_1': 'PIS With Particle', 'glass_type': 'ALL', 'defect_size': 'M', 'glass_count': 55.0, 'defect_count': 138.0, 'density': '2.51', 'first_over_std': '7.53', 'std': '1.53', 'ooc': '7.09', 'oos': '11.6

KeyboardInterrupt: 

In [26]:
match_fail

{1785: {'cl': 'glass_count',
  'val': '397',
  'data': {'line_id': 'CAPIC200',
   'aoi': 'aoi100',
   'model': 'M195RTN01',
   'recipe_id': '803',
   'glass_type': 'ALL',
   'ai_code_1': 'PI_Spot_NP',
   'defect_size': 'S',
   'glass_count': '397',
   'defect_count': '2137',
   'density': '5.38',
   'first_over_std': '16.15',
   'std': '3.01',
   'ooc': '14.41',
   'oos': '23.44'}},
 1786: {'cl': 'glass_count',
  'val': '381',
  'data': {'line_id': 'CAPIC200',
   'aoi': 'aoi100',
   'model': 'M195RTN01',
   'recipe_id': '803',
   'glass_type': 'ALL',
   'ai_code_1': 'PI_Spot_NP',
   'defect_size': 'M',
   'glass_count': '381',
   'defect_count': '1758',
   'density': '4.61',
   'first_over_std': '13.84',
   'std': '2.69',
   'ooc': '12.69',
   'oos': '20.77'}},
 1787: {'cl': 'glass_count',
  'val': '40',
  'data': {'line_id': 'CAPIC200',
   'aoi': 'aoi100',
   'model': 'M195RTN01',
   'recipe_id': '803',
   'glass_type': 'ALL',
   'ai_code_1': 'PI_Spot_NP',
   'defect_size': 'L',
   'g

In [ ]:
### 動態density spec計算
#
def density_spec_calculate(rows):
    rows['density'] = rows[ 'n_rows']/rows[ 'n_glasses']
    density_mean = rows['density'].mean()
    density_std = rows['density'].std()
    
    ooc = density_mean + 3 * density_std
    oos = density_mean + 6 * density_std

    return [round(v,1) for v in [density_mean, density_std, ooc, oos]]

group_keys = ['line_id', 'aoi', 'model', 'ai_code_1']
defect_codes = ['Polymer', 'SSIU_Polymer', 'PI_Spot_NP', 'PIS With Particle']

df = pd.DataFrame()
for tbn in ['pidensity_202509','pidensity_202510']:
    tb = dbhandler.get_table(tbn)
    df = pd.concat([df, tb])
   

def robust_density_stats(series: pd.Series, trim_mult: float = 3.0):
    """
    

    規則：先用所有值算第一次平均 m1，剔除所有 > trim_mult*m1 的值，再用剩下的值計算
          mean / std（ddof=1）/ OOC=mean+3*std / OOS=mean+6*std。
    皆四捨五入到小數點一位。
    邊界處理：
      - 全為 NaN → 全回 0
      - 有效樣本數=1 → std=0
      - 第一次平均 m1 <= 0 → 不做剔除（避免門檻=0 把全部正值剔除）
    """
    s = pd.to_numeric(series, errors='coerce').replace([np.inf, -np.inf], np.nan).dropna()
    if s.empty:
        return (0.0, 0.0, 0.0, 0.0)

    m1 = float(s.mean())
    if m1 > 0:
        thresh = trim_mult * m1
        s2 = s[s <= thresh]
        # 若全被剔除則回退到原始 s
        if s2.empty:
            s2 = s
    else:
        s2 = s  # m1<=0 時不剔除

    mean = float(s2.mean())
    std  = float(s2.std(ddof=1))
    if np.isnan(std):
        std = 0.0

    ooc = mean + 3.0 * std
    oos = mean + 6.0 * std
    return (round(mean, 1), round(std, 1), round(ooc, 1), round(oos, 1))

vals = robust_density_stats(group_rows['density'])
vals
def _density_stats(series: pd.Series) :
    s = series.dropna()
    if s.empty:
        return (0.0, 0.0, 0.0, 0.0)
    mean = float(s.mean())
    print(s.values)
    std  = float(s.std(ddof=1))
    if pd.isna(std):
        std = 0.0
    ooc = mean + 3.0 * std
    oos = mean + 6.0 * std
    return (round(mean, 1), round(std, 1), round(ooc, 1), round(oos, 1))

vals = _density_stats(group_rows['density'])
vals
if not df.empty:
    df = df[df['ai_code_1'].isin(defect_codes)]
    spec_dict = []
    group_num= 0
    for groups, group_rows in df.groupby(group_keys):
        line, aoi, model, defect_code = groups
        group_num +=1
        #glass side不分群
        density_mean, density_std, ooc, oos = density_spec_calculate(group_rows)
        spec_dict.append([line, aoi, model, defect_code, 'all', density_mean, density_std, ooc, oos, len(group_rows)])
        print(line, aoi, model, defect_code, len(group_rows))
        
        print(density_mean, density_std, ooc, oos)
        #if density_std ==0 or density_mean ==1:
        #print(group_rows)
        #glass side分群
        for side, side_rows in group_rows.groupby(['glass_type']):
            density_mean, density_std, ooc, oos = density_spec_calculate(side_rows)
            spec_dict.append([line, aoi, model, defect_code, side[0], density_mean, density_std, ooc, oos, len(side_rows)])
    spec_df = pd.DataFrame(spec_dict,columns = ['PI Line', 'AOI', 'Model', 'defect code', 'glass side', 'AVG', 'STD', 'OOC', 'OOS', 'row_num'])
print( group_num , len(df))
#spec_df.to_csv('spec_calculate2.csv')

In [ ]:

# -------- 小工具 --------
def _parse_dt(s: str) -> datetime:
    s = s.strip().replace("T", " ")
    fmts = ["%Y-%m-%d %H:%M:%S", "%Y-%m-%d %H:%M", "%Y-%m-%d"]
    for f in fmts:
        try:
            return datetime.strptime(s, f)
        except ValueError:
            continue
    raise ValueError(f"Bad datetime: {s}")

def _month_span(start: datetime, end: datetime) -> List[str]:
    ym = []
    cur = datetime(start.year, start.month, 1)
    last = datetime(end.year, end.month, 1)
    while cur <= last:
        ym.append(cur.strftime("%Y%m"))
        # 下一個月
        nxt = (cur.replace(day=28) + timedelta(days=4)).replace(day=1)
        cur = nxt
    return ym

def _try_get_table(dbhandler, tbn: str) -> Optional[pd.DataFrame]:
    try:
        df = dbhandler.get_table(tbn)
        if df is not None and len(df) > 0:
            return df
    except Exception:
        pass
    # 試試大寫
    try:
        df = dbhandler.get_table(tbn.upper())
        if df is not None and len(df) > 0:
            return df
    except Exception:
        pass
    return None

def _apply_filters(df: pd.DataFrame, filters: Dict[str, List[str]]) -> pd.DataFrame:
    if df.empty or not filters:
        return df

    df2 = df.copy()

    # 通用等值過濾
    for key in ["line_id", "aoi", "model", "glass_type", "ai_code_1", "pi", "recipe_id", "defect_type"]:
        if key in filters and filters[key]:
            if key in df2.columns:
                df2 = df2[df2[key].isin(filters[key])]

    # pi_hour：支援 小時 等值 / 日期整天 / 區間(~、|、,)
    if "pi_hour" in filters and filters["pi_hour"]:
        df2["pi_hour"] = pd.to_datetime(df2["pi_hour"])
        parts = []
        for raw in filters["pi_hour"]:
            if not isinstance(raw, str) or not raw.strip():
                continue
            s = raw.strip()
            # 區間
            if any(sep in s for sep in ("~", "|", ",")):
                for sep in ("~", "|", ","):
                    if sep in s:
                        a, b = [x.strip() for x in s.split(sep, 1)]
                        break
                start = _parse_dt(a)
                end   = _parse_dt(b)
                if end < start:
                    start, end = end, start
                # 區間含當小時
                m = (df2["pi_hour"] >= start) & (df2["pi_hour"] <= end)
                parts.append(m)
            else:
                # 單點：日期或小時
                dt = _parse_dt(s)
                if len(s) <= 10:  # 只有 YYYY-MM-DD
                    day0 = datetime(dt.year, dt.month, dt.day)
                    day1 = day0 + timedelta(days=1) - timedelta(seconds=1)
                    m = (df2["pi_hour"] >= day0) & (df2["pi_hour"] <= day1)
                else:
                    m = (df2["pi_hour"] == dt)
                parts.append(m)
        if parts:
            mask = parts[0]
            for m in parts[1:]:
                mask = mask | m
            df2 = df2[mask]

    # glass_id：聚合字串包含任一
    if "glass_id" in filters and filters["glass_id"] and "glass" in df2.columns:
        wanted = set([s.strip() for s in filters["glass_id"] if s and isinstance(s, str)])
        if wanted:
            def has_any_glass(cell: Any) -> bool:
                if pd.isna(cell) or not str(cell).strip():
                    return False
                items = [x.strip() for x in str(cell).split(",") if x.strip()]
                return bool(set(items) & wanted)
            df2 = df2[df2["glass"].apply(has_any_glass)]

    # defect_size：S/M/L/O 任何一種 >0
    if "defect_size" in filters and filters["defect_size"]:
        m = {"S": "small_defect_count", "M": "middle_defect_count",
             "L": "large_defect_count", "O": "over_defect_count"}
        cols = [m[s] for s in filters["defect_size"] if s in m and m[s] in df2.columns]
        if cols:
            df2 = df2[df2[cols].fillna(0).sum(axis=1) > 0]

    return df2


def _sorted_unique_glasses(glass_cell: Any) -> List[str]:
    if pd.isna(glass_cell) or not str(glass_cell).strip():
        return []
    items = [x.strip() for x in str(glass_cell).split(",") if x.strip()]
    return sorted(set(items))

def _build_chart_dict(df: pd.DataFrame) -> Dict[str, Any]:
    """
    結構：
    chart[line_id][aoi][model][glass_type][ai_code_1][pi_hour_str] = {
        'defect_type': ..,
        'n_rows': ..,
        'n_glasses': ..,       # 以 glass 去重後的計數
        'small_defect_count': ...,
        'middle_defect_count': ...,
        'large_defect_count': ...,
        'over_defect_count': ...,
        'unknown_defect_count': ...,
        'glass': 'g1,g2,...'   # 去重後排序串接
    }
    * 若同一分群同一 pi_hour 有多列（例如不同 recipe_id），自動合併數值並 glass 取聯集。
    """
    if df.empty:
        return {}

    # 確保型別
    work = df.copy()
    work["pi_hour"] = pd.to_datetime(work["pi_hour"])
    for c in ["n_rows", "small_defect_count", "middle_defect_count", "large_defect_count",
              "over_defect_count", "unknown_defect_count"]:
        if c in work.columns:
            work[c] = pd.to_numeric(work[c], errors="coerce").fillna(0).astype(int)
    # 準備容器
    chart: Dict[str, Any] = {}

    # 我們把 recipe 維度併掉（你要的巢狀鍵不包含 recipe_id）
    key_cols = ["line_id", "aoi", "model", "glass_type", "ai_code_1", "defect_type", "pi_hour"]

    # 逐列累加
    for _, r in work.iterrows():
        line_id = r.get("line_id", "")
        aoi     = r.get("aoi", "")
        model   = r.get("model", "")
        gside   = r.get("glass_type", "")
        code    = r.get("ai_code_1", "") or ""
        dtype   = r.get("defect_type", "") or ""
        ts      = r["pi_hour"]
        ts_key  = ts.strftime("%Y-%m-%d %H:%M:%S")

        glist = _sorted_unique_glasses(r.get("glass", ""))

        # 取到節點
        d1 = chart.setdefault(line_id, {})
        d2 = d1.setdefault(aoi, {})
        d3 = d2.setdefault(model, {})
        d4 = d3.setdefault(gside, {})
        d5 = d4.setdefault(code, {})

        node = d5.get(ts_key)
        if node is None:
            d5[ts_key] = {
                "defect_type": dtype,
                "n_rows": int(r.get("n_rows", 0)),
                "n_glasses": len(glist),  # 先用當列的，後面合併會更新
                "small_defect_count": int(r.get("small_defect_count", 0)),
                "middle_defect_count": int(r.get("middle_defect_count", 0)),
                "large_defect_count": int(r.get("large_defect_count", 0)),
                "over_defect_count": int(r.get("over_defect_count", 0)),
                "unknown_defect_count": int(r.get("unknown_defect_count", 0)),
                "_glass_set": set(glist),   # 內部用，最後會轉字串
            }
        else:
            node["n_rows"] += int(r.get("n_rows", 0))
            node["small_defect_count"]  += int(r.get("small_defect_count", 0))
            node["middle_defect_count"] += int(r.get("middle_defect_count", 0))
            node["large_defect_count"]  += int(r.get("large_defect_count", 0))
            node["over_defect_count"]   += int(r.get("over_defect_count", 0))
            node["unknown_defect_count"]+= int(r.get("unknown_defect_count", 0))
            node["_glass_set"].update(glist)

    # 收尾：把 _glass_set 轉為 glass 串、並更新 n_glasses
    def _finalize(d: Any):
        if isinstance(d, dict):
            keys = list(d.keys())
            for k in keys:
                v = d[k]
                if isinstance(v, dict) and "_glass_set" in v:
                    gset = v.pop("_glass_set")
                    glass_list = sorted(gset)
                    d[k]["glass"] = ",".join(glass_list)
                    d[k]["n_glasses"] = len(glass_list)
                else:
                    _finalize(v)
    _finalize(chart)
    return chart


In [ ]:
summary = dbhandler.get_table('pidensity_202509')
#summary[(summary['aoi'] == 'AOI200') & (summary['line_id'] == 'CAPIC200') & (summary['model'] == 'M270DANA0')]
summary.head(3)

In [ ]:
# ===== defect map helpers =====
def _hour_bounds(ht: datetime) -> (datetime, datetime):
    start = datetime(ht.year, ht.month, ht.day, ht.hour, 0, 0)
    end = start + timedelta(hours=1)
    return start, end

def _size_bucket(v) -> Optional[str]:
    try:
        x = int(v)
    except Exception:
        return None
    if x >= 401: return "O"
    if 101 <= x <= 400: return "L"
    if 21  <= x <= 100: return "M"
    if x <= 20: return "S"
    return None

def _filter_raw_for_group(raw: pd.DataFrame, row: Dict[str, Any],
                          glass_ids: List[str],
                          size_wanted: Optional[List[str]] = None) -> pd.DataFrame:
    if raw is None or raw.empty:
        return raw.iloc[0:0]

    r = row
    # 標準欄位準備
    raw = raw.copy()
    
    if "scan_time" in raw.columns:
        raw["scan_time"] = pd.to_datetime(raw["scan_time"], errors="coerce")
        raw["pi_hour_calc"] = raw["scan_time"].dt.floor("H")
    elif "dayhour" in raw.columns:
        # 允許 'YYYY-MM-DD HH' 或 'YYYY-MM-DD HH:00:00'
        raw["pi_hour_calc"] = pd.to_datetime(raw["dayhour"].astype(str).str.replace(r"(\d{4}-\d{2}-\d{2}\s\d{2})(?!:)", r"\1:00:00", regex=True),
                                             errors="coerce")
    elif {"day","hour"}.issubset(set(raw.columns)):
        raw["pi_hour_calc"] = pd.to_datetime(raw["day"].astype(str) + " " + raw["hour"].astype(str).str.zfill(2) + ":00:00",
                                             errors="coerce")
    else:
        raw["pi_hour_calc"] = pd.NaT
    #print(raw)
    # 各式條件
    pi_hour = pd.to_datetime(r["pi_hour"])
    aoi = str(r["aoi"]).upper()
    line_id = r["line_id"]
    model = r["model"]
    gside = r["glass_type"]
    recipe_pref = str(r["recipe_id"])
    ai1 = (r.get("ai_code_1") or "").strip()
    #print(pi_hour)


    # 比對欄位
    mask = pd.Series(True, index=raw.index)
    if "line_id" in raw:    mask &= (raw["line_id"] == line_id)
    if "model" in raw:      mask &= (raw["model"] == model)
    if "glass_type" in raw: mask &= (raw["glass_type"] == gside)

    # recipe_id prefix 比對（LEFT(recipe_id, len)==prefix）
    if "recipe_id" in raw:
        mask &= raw["recipe_id"].astype(str).str[:len(recipe_pref)] == recipe_pref

    # ai_code_1 清理比對
    if "ai_code_1" in raw:
        raw["_ai1"] = raw["ai_code_1"].fillna("").astype(str).str.strip()
        mask &= (raw["_ai1"] == ai1)

    # 小時比對
    mask &= (raw["pi_hour_calc"] == pi_hour)
    #print(f'mask:{mask}')
    # glass 限定
    if "glass_id" in raw:
        mask &= raw["glass_id"].astype(str).isin(glass_ids)

    out = raw[mask].copy()
    #print(out)
    # defect_size 過濾（S/M/L/O）
    if size_wanted:
        out["_bucket"] = out["defect_size"].apply(_size_bucket)
        out = out[out["_bucket"].isin(size_wanted)]

    return out


In [ ]:
tables = dbhandler.list_tables()
tables

In [ ]:
for tbn in tables:
    if '_pidensity_' in tbn and '202510' in tbn:
        print(tbn)
        tb = dbhandler.get_table(tbn)
        if not tb.empty:
            print(tb.iloc[0,:].to_dict())
            break

In [ ]:
tb.iloc[0,:].to_dict()

In [ ]:
#dbhandler.list_tables()
tbn =  'aoi_summary_aoi200'#'aoi_spec_for_aoimonitor'#'pidensity_202509'#'aoi_density_for_aoimonitor' #'aoi200_pidensity_202509_pi200'
df = dbhandler.get_table(tbn)
df.iloc[-1,:].to_dict()

In [ ]:
cfg = Config()
dbhandler = MySQLConnet('l6a01_project')

try:
    filters: Dict[str, List[str]] = filter_ask_keys if filter_ask_keys else {}
    if not isinstance(filters, dict):
        filters = {}
except Exception:
    filters = {}

months: List[str] = []
if "pi_hour" in filters and filters["pi_hour"]:
    # 找到最小與最大的時間以覆蓋跨月
    mins, maxs = [], []
    for s in filters["pi_hour"]:
        if not isinstance(s, str) or not s.strip():
            continue
        if any(sep in s for sep in ("~", "|", ",")):
            for sep in ("~", "|", ","):
                if sep in s:
                    a, b = [x.strip() for x in s.split(sep, 1)]
                    break
            mins.append(_parse_dt(a)); maxs.append(_parse_dt(b))
        else:
            dt = _parse_dt(s)
            if len(s.strip()) <= 10:  # 只有日期
                day0 = datetime(dt.year, dt.month, dt.day)
                day1 = day0 + timedelta(days=1) - timedelta(seconds=1)
                mins.append(day0); maxs.append(day1)
            else:
                mins.append(dt);  maxs.append(dt)
    if mins and maxs:
        start, end = min(mins), max(maxs)
        months = _month_span(start, end)

if not months:
    now = datetime.now()
    months = [now.strftime("%Y%m")]
print(months)

frames: List[pd.DataFrame] = []
for ym in months:
    tbn = f"pidensity_{ym}"
    df = _try_get_table(dbhandler, tbn)
    if df is None or df.empty:
        continue
    keep_cols = [c for c in cfg.aoi_density_summary_sql_cols if c in df.columns]
    df = df[keep_cols].copy()
    df["pi_hour"] = pd.to_datetime(df["pi_hour"])
    frames.append(df)

if frames:
    df_all = pd.concat(frames, ignore_index=True)
else:
    print(f'return DefectGroupDict: none, ParamDict: cfg.front_config')

# 先用 filters 過 summary
df_filtered = _apply_filters(df_all, filters)

size_wanted = None
if "defect_size" in filters and filters["defect_size"]:
    allow = {"S", "M", "L", "O"}
    size_wanted = [s for s in filters["defect_size"] if s in allow]
    if not size_wanted:
        size_wanted = None
print(f'size_wanted:{size_wanted}')



In [ ]:
# 讀 raw 表快取
raw_cache: Dict[str, pd.DataFrame] = {}

def _get_raw(aoi: str, pi: str, ym: str) -> Optional[pd.DataFrame]:
    tname = f"{aoi.lower()}_pidensity_{ym}_{pi.lower()}"
    print(tname)
    if tname not in raw_cache:
        tb =  _try_get_table(dbhandler, tname) #or pd.DataFrame()
        #print(tb.iloc[:2,:])
        raw_cache[tname] = tb#.iloc[:2,:]
    return raw_cache[tname]

# 輸出結構：line_id → aoi → model → glass_type → ai_code_1 → pi_hour → { ... }
result: Dict[str, Any] = {}
for _, r in df_filtered.iterrows():
    
    pi_hour = pd.to_datetime(r["pi_hour"])
    ym = pi_hour.strftime("%Y%m")
    aoi = str(r["aoi"]).upper()
    pi  = str(r["pi"]).upper()

    raw = _get_raw(aoi, pi, ym)
    if raw is None or raw.empty:
        # 試試大寫表名（避免大小寫差異）
        tname2 = f"{aoi}_pidensity_{ym}_{pi}"
        if tname2 not in raw_cache:
            raw_cache[tname2] = _try_get_table(dbhandler, tname2) or pd.DataFrame()
        raw = raw_cache[tname2]
        
        if raw is None or raw.empty:
            continue

    glass_ids = _sorted_unique_glasses(r.get("glass", ""))
    #print(f'{r.tolist()}\n', glass_ids, size_wanted)
    #print(raw)
    sub = _filter_raw_for_group(raw, r, glass_ids, size_wanted=size_wanted)
    if sub.empty:
        continue
    
    # 分 glass 統計 + 缺陷點位
    sub["_bucket"] = sub["defect_size"].apply(_size_bucket)
    print(sub)

    by_glass = {}
    for gid, gdf in sub.groupby(sub["glass_id"].astype(str)):
        s = int((gdf["_bucket"] == "S").sum())
        m = int((gdf["_bucket"] == "M").sum())
        l = int((gdf["_bucket"] == "L").sum())
        o = int((gdf["_bucket"] == "O").sum())
        rows = gdf[["x", "y", "chip_name", "pic_name", "defect_size"]].fillna("").to_dict(orient="records")
        by_glass[gid] = {
            "glass_id": gid,
            "glass_defect_count": len(gdf),
            "small_defect_count": s,
            "middle_defect_count": m,
            "large_defect_count": l,
            "over_defect_count": o,
            "rows": rows
        }

    # 塞入巢狀結構
    line_id = r["line_id"]; model = r["model"]; gside = r["glass_type"]
    code    = (r.get("ai_code_1") or "").strip()
    dtype   = (r.get("defect_type") or "").strip()
    ts_key  = pi_hour.strftime("%Y-%m-%d %H:%M:%S")

    d1 = result.setdefault(line_id, {})
    d2 = d1.setdefault(aoi, {})
    d3 = d2.setdefault(model, {})
    d4 = d3.setdefault(gside, {})
    d5 = d4.setdefault(code, {})
    d6 = d5.setdefault(ts_key, {"defect_type": dtype, "glasses": {}})

    # 合併同 group 的 glass（避免同 group 多列）
    for gid, payload in by_glass.items():
        if gid in d6["glasses"]:
            # 合併數量與 rows
            old = d6["glasses"][gid]
            old["glass_defect_count"] += payload["glass_defect_count"]
            old["small_defect_count"]  += payload["small_defect_count"]
            old["middle_defect_count"] += payload["middle_defect_count"]
            old["large_defect_count"]  += payload["large_defect_count"]
            old["over_defect_count"]   += payload["over_defect_count"]
            old["rows"].extend(payload["rows"])
        else:
            d6["glasses"][gid] = payload

In [ ]:

import pandas as pd
import logging
from sqlalchemy import text, inspect
from sqlalchemy.exc import SQLAlchemyError
from datetime import datetime, timedelta
import re

def get_defects_by_key(db, table: str, key_dict: dict):
    """
    依 key_dict 動態篩選資料（key 為表內任意欄位；值可為單值或可迭代 -> IN）。
    特別鍵：pi_hour
      - 可為單值或可迭代（多小時）
      - 支援格式：'YY-MM-DD HH'（e.g. '25-09-30 10'）、'YYYY-MM-DD HH'、'YYYY-MM-DD HH:MM'、'YYYY-MM-DD HH:MM:SS'
      - 會轉成 [scantime] 在該小時區間 (含起不含迄)
    回傳：
      {
        "<glass_id>": {
          "defect_map": [ {x,y,size,img,chip,type?,recipe_id?}, ... ],
          "S": s_count, "M": m_count, "L": l_count, "O": o_count, "total": s+m+l+o
        },
        ...
      }
    """
    # --- 檢查 table 名 ---
    if not re.match(r"^[A-Za-z_][A-Za-z0-9_]*$", table or ""):
        raise ValueError(f"Bad table name: {table}")

    # 取欄位清單
    try:
        insp = inspect(db.engine)
        cols = set(c["name"] for c in insp.get_columns(table))
    except Exception:
        cols = None  # 若取不到，就寬鬆處理（仍有正則保護欄位名）

    # 找時間欄位（優先 scantime，否則 scan_time）
    time_col = None
    for cand in ("scantime", "scan_time"):
        if cols is None or cand in cols:
            time_col = cand
            break
    if time_col is None:
        raise ValueError(f"Table '{table}' must contain a time column 'scantime' (or 'scan_time').")

    # 解析 pi_hour -> (start, end)
    def parse_pi_hour_one(val):
        s = str(val).strip()
        # 25-09-30 10 -> 2025-09-30 10:00:00
        if re.match(r"^\d{2}-\d{2}-\d{2}\s+\d{2}$", s):
            yy, mm, dd = map(int, s.split()[0].split("-"))
            hh = int(s.split()[1])
            start = datetime(2000 + yy, mm, dd, hh, 0, 0)
            end = start + timedelta(hours=1)
            return start, end
        # 其他常見格式
        for fmt in ("%Y-%m-%d %H", "%Y-%m-%d %H:%M", "%Y-%m-%d %H:%M:%S"):
            try:
                dt = datetime.strptime(s, fmt)
                start = dt.replace(minute=0, second=0, microsecond=0)
                end = start + timedelta(hours=1)
                return start, end
            except ValueError:
                pass
        raise ValueError(f"Bad pi_hour format: {val}")

    # --- 動態 WHERE ---
    where_parts, params = [], {}

    def _is_seq(x):
        return isinstance(x, (list, tuple, set))

    if not key_dict:
        logging.warning("[get_defects_by_key] key_dict is empty; return empty to avoid full scan.")
        return {}

    # 先抽出 pi_hour，其餘照舊
    pi_val = key_dict.pop("pi_hour", None)

    # 其他一般欄位
    for k, v in key_dict.items():
        if not re.match(r"^[A-Za-z_][A-Za-z0-9_]*$", str(k)):
            raise ValueError(f"Bad column name: {k}")
        if cols is not None and k not in cols:
            raise ValueError(f"Column '{k}' not found in table '{table}'")

        if _is_seq(v):
            vv = list(v)
            if not vv:
                where_parts.append("1=0")
                continue
            phs = []
            for i, item in enumerate(vv):
                pname = f"k_{k}_{i}"
                phs.append(f":{pname}")
                params[pname] = item
            where_parts.append(f"`{k}` IN ({', '.join(phs)})")
        else:
            pname = f"k_{k}"
            where_parts.append(f"`{k}` = :{pname}")
            params[pname] = v

    # 加入 pi_hour 篩選（支援單值/多值 → OR ）
    if pi_val is not None:
        if _is_seq(pi_val):
            sub = []
            for i, item in enumerate(list(pi_val)):
                st, en = parse_pi_hour_one(item)
                p0, p1 = f"ph_{i}_0", f"ph_{i}_1"
                params[p0], params[p1] = st, en
                sub.append(f"(`{time_col}` >= :{p0} AND `{time_col}` < :{p1})")
            if sub:
                where_parts.append("(" + " OR ".join(sub) + ")")
        else:
            st, en = parse_pi_hour_one(pi_val)
            params["ph_0"], params["ph_1"] = st, en
            where_parts.append(f"`{time_col}` >= :ph_0 AND `{time_col}` < :ph_1")

    where_sql = " AND ".join(where_parts) if where_parts else "1=1"

    # SELECT 欄位
    if cols is not None and "glass_id" not in cols:
        raise ValueError(f"Table '{table}' must contain column 'glass_id' for grouping.")

    select_fields = []
    def add_if(colname, expr):
        if cols is None or colname in cols:
            select_fields.append(expr)

    add_if("x",           "CAST(x AS UNSIGNED) AS x")
    add_if("y",           "CAST(y AS UNSIGNED) AS y")
    add_if("defect_size", "defect_size         AS size")
    add_if("pic_name",    "pic_name            AS img")
    add_if("chip_name",   "chip_name           AS chip")
    add_if("defect_type", "defect_type         AS type")
    add_if("recipe_id",   "recipe_id           AS recipe_id")
    add_if("glass_id",    "glass_id")

    if not any(f.endswith("glass_id") or " AS glass_id" in f for f in select_fields):
        select_fields.append("glass_id")

    sql = f"""
        SELECT
            {', '.join(select_fields)}
        FROM `{table}`
        WHERE {where_sql}
    """

    try:
        with db.engine.begin() as conn:
            res = conn.execute(text(sql), params).mappings().all()
            rows = [dict(r) for r in res]
    except SQLAlchemyError as e:
        logging.error(f"[get_defects_by_key] {e}")
        return {}

    # 分組並計算 S/M/L/O/total
    def to_bucket(val):
        if val is None:
            return None
        if isinstance(val, str):
            up = val.strip().upper()
            if up in ("S","M","L","O"):
                return up
            try:
                num = int(float(up))
            except Exception:
                return None
        else:
            try:
                num = int(float(val))
            except Exception:
                return None
        if num <= 20: return "S"
        if 21 <= num <= 100: return "M"
        if 101 <= num <= 400: return "L"
        if num >= 401: return "O"
        return None

    grouped = {}
    for r in rows:
        gid = str(r.get("glass_id", "") or "")
        item = {k: v for k, v in r.items() if k != "glass_id"}
        g = grouped.setdefault(gid, {"defect_map": [], "S": 0, "M": 0, "L": 0, "O": 0, "total": 0})
        g["defect_map"].append(item)
        b = to_bucket(item.get("size"))
        if b in ("S","M","L","O"):
            g[b] += 1
            g["total"] += 1

    # 把 pi_hour 放回 key_dict 以免外層需要（不影響本函式主要輸出）
    if pi_val is not None:
        key_dict["pi_hour"] = pi_val

    return grouped



In [25]:
front_rows =  [{'pi_hour': '25-11-05 14', 'aoi': 'aoi200', 'pi': '', 'line_id': 'CAPIC600', 'model': 'B160UAN4A', 'glass_type': 'TFT', 'recipe_id': '2251', 'recipe_comment': 'B160UAN4A-T-D-API', 'defect_type': '', 'ai_code_1': 'PI_Gel', 'maingroup_glass_count': 1, 'defect_code_count': 1, 'maingroup_defect_count': 14, 'small_defect_count': 1, 'middle_defect_count': 0, 'large_defect_count': 0, 'over_defect_count': 0, 'n_glasses': 1, 'n_rows': 1, 'density': 1, 'glass': '5H5AASY4D', 'glass_defect_count': {}}]
print(f'group count: {len(front_rows)}')
rawdata_keys = ['line_id', 'model', 'glass_type', 'recipe_id', 'ai_code_1', 'pi_hour']
rawdata_keys = ['pi_hour', 'line_id', 'model', 'glass_type', 'recipe_id', 'recipe_comment', 'ai_code_1']

group count: 1


In [37]:
from sqlalchemy import text, bindparam
from sqlalchemy.exc import SQLAlchemyError
import re
import logging

def get_defects_by_key(self, table: str, key_dict: dict):
    """
    依 key_dict 產生多欄位條件：
      - 單值: '='
      - list/tuple/set: 'IN (...)'
      - None: IS NULL
    只使用實際存在於資料表的欄位，並回傳完整列資料（list[dict]）。
    """
    if not key_dict:
        return []

    # 簡單防呆：表名只允許英數與底線
    if not re.fullmatch(r"[A-Za-z0-9_]+", table or ""):
        logging.error(f"[get_defects_by_key] illegal table name: {table!r}")
        return []

    # 取得該表實際欄位
    try:
        with self.engine.begin() as conn:
            cols = conn.execute(
                text("""
                    SELECT COLUMN_NAME
                    FROM INFORMATION_SCHEMA.COLUMNS
                    WHERE TABLE_SCHEMA = :db AND TABLE_NAME = :tbl
                """),
                {"db": getattr(self, "db", None), "tbl": table},
            ).scalars().all()
    except Exception:
        # 若 INFORMATION_SCHEMA 無法用，退回 SHOW COLUMNS
        try:
            with self.engine.begin() as conn:
                rows = conn.execute(text(f"SHOW COLUMNS FROM `{table}`")).all()
            cols = [r[0] for r in rows]  # 第一欄為欄名
        except Exception as e:
            logging.error(f"[get_defects_by_key] fetch columns failed: {e}")
            return []

    colset = set(cols)
    if not colset:
        return []

    where, params, expanding = [], {}, []

    for k, v in key_dict.items():
        if k not in colset:
            continue  # 忽略不存在的欄位
        col = f"`{k}`"

        if v is None:
            where.append(f"{col} IS NULL")
        elif isinstance(v, (list, tuple, set)):
            lst = list(v)
            if not lst:
                # 空 IN 沒意義，跳過此條件
                continue
            pname = f"{k}_list"
            where.append(f"{col} IN :{pname}")
            params[pname] = lst
            expanding.append(bindparam(pname, expanding=True))
        else:
            where.append(f"{col} = :{k}")
            params[k] = v

    if not where:
        return []  # 沒有任何有效條件，避免掃全表

    sql = f"SELECT * FROM `{table}` WHERE " + " AND ".join(where)

    try:
        with self.engine.begin() as conn:
            stmt = text(sql)
            if expanding:
                stmt = stmt.bindparams(*expanding)
            rows = conn.execute(stmt, params).mappings().all()
            return [dict(r) for r in rows]   # 回傳完整列
    except SQLAlchemyError as e:
        logging.error(f"[get_defects_by_key] {e}")
        return []

In [ ]:

all_defect_groups = {}
return_dict = []
for filters in front_rows :
    print(filters)
    print(filters.keys())
    aoi, ym, pi = filters['aoi'], filters['pi_hour'][:5].replace("-",''), filters['line_id'][-3]
    key_dict = {k: v for k, v in filters.items() if k in rawdata_keys }
    key_dict['pi_hour'] = key_dict['pi_hour'] if len(key_dict['pi_hour'] ) ==13 else  '20'+key_dict['pi_hour']
    
    tbname = f'{aoi}_pidensity_20{ym}_pi{pi}00'
    print(tbname, key_dict)
    defect_rows = get_defects_by_key(dbhandler, tbname, key_dict)
    print(f'match defect data: {len(defect_rows)}')
    print(pd.DataFrame(defect_rows))
#print(return_dict)


    

{'pi_hour': '25-11-05 14', 'aoi': 'aoi200', 'pi': '', 'line_id': 'CAPIC600', 'model': 'B160UAN4A', 'glass_type': 'TFT', 'recipe_id': '2251', 'recipe_comment': 'B160UAN4A-T-D-API', 'defect_type': '', 'ai_code_1': 'PI_Gel', 'maingroup_glass_count': 1, 'defect_code_count': 1, 'maingroup_defect_count': 14, 'small_defect_count': 1, 'middle_defect_count': 0, 'large_defect_count': 0, 'over_defect_count': 0, 'n_glasses': 1, 'n_rows': 1, 'density': 1, 'glass': '5H5AASY4D', 'glass_defect_count': {}}
dict_keys(['pi_hour', 'aoi', 'pi', 'line_id', 'model', 'glass_type', 'recipe_id', 'recipe_comment', 'defect_type', 'ai_code_1', 'maingroup_glass_count', 'defect_code_count', 'maingroup_defect_count', 'small_defect_count', 'middle_defect_count', 'large_defect_count', 'over_defect_count', 'n_glasses', 'n_rows', 'density', 'glass', 'glass_defect_count'])
aoi200_pidensity_202511_pi600 {'pi_hour': '2025-11-05 14', 'line_id': 'CAPIC600', 'model': 'B160UAN4A', 'glass_type': 'TFT', 'recipe_id': '2251', 'reci

In [ ]:
http://10.97.139.98:1454/CAAOI202/P4G32AA0509/BN5A9VL5J/KOUID/20251105082415/'+ ' CaptureImage/RV1_1368927_552368_98' + '.JPG'
http://10.97.139.98:1454/CAPIT203/CA0260/AE1U4A09Y0/CAPIC300/20251104113500//CaptureImage/RV1_1034506_235446_39.jpg

In [ ]:

@router.get("/api/defect_map")
async def defect_map(
    filter_ask_keys: Optional[str] = Query(None, description="JSON 物件字串：{'line_id': [...], 'aoi': [...], 'model': [...], 'glass_type': [...], 'ai_code_1': [...], 'glass_id': [...], 'defect_size': ['S','M','L','O'], 'pi_hour': [...]}")
):
    cfg = Config()
    dbhandler = MySQLConnet('l6a01_project')

    # 解析 filters
    try:
        filters: Dict[str, List[str]] = json.loads(filter_ask_keys) if filter_ask_keys else {}
        if not isinstance(filters, dict):
            filters = {}
    except Exception:
        filters = {}

    # 依 pi_hour 決定要載入哪些月份的 summary 表；若沒給就用當月
    months: List[str] = []
    if "pi_hour" in filters and filters["pi_hour"]:
        # 找到最小與最大的時間以覆蓋跨月
        mins, maxs = [], []
        for s in filters["pi_hour"]:
            if not isinstance(s, str) or not s.strip():
                continue
            if any(sep in s for sep in ("~", "|", ",")):
                for sep in ("~", "|", ","):
                    if sep in s:
                        a, b = [x.strip() for x in s.split(sep, 1)]
                        break
                mins.append(_parse_dt(a)); maxs.append(_parse_dt(b))
            else:
                dt = _parse_dt(s)
                if len(s.strip()) <= 10:  # 只有日期
                    day0 = datetime(dt.year, dt.month, dt.day)
                    day1 = day0 + timedelta(days=1) - timedelta(seconds=1)
                    mins.append(day0); maxs.append(day1)
                else:
                    mins.append(dt);  maxs.append(dt)
        if mins and maxs:
            start, end = min(mins), max(maxs)
            months = _month_span(start, end)

    if not months:
        now = datetime.now()
        months = [now.strftime("%Y%m")]

    # 載入 summary（合併跨月）
    frames: List[pd.DataFrame] = []
    for ym in months:
        tbn = f"pidensity_{ym}"
        df = _try_get_table(dbhandler, tbn)
        if df is None or df.empty:
            continue
        keep_cols = [c for c in cfg.aoi_density_summary_sql_cols if c in df.columns]
        df = df[keep_cols].copy()
        df["pi_hour"] = pd.to_datetime(df["pi_hour"])
        frames.append(df)

    if frames:
        df_all = pd.concat(frames, ignore_index=True)
    else:
        return {"DefectGroupDict": {}, "ParamDict": cfg.front_config}

    # 先用 filters 過 summary
    df_filtered = _apply_filters(df_all, filters)

    if df_filtered.empty:
        return {"DefectGroupDict": {}, "ParamDict": cfg.front_config}

    # 解析欲保留的 defect_size（用在 raw 過濾）
    size_wanted = None
    if "defect_size" in filters and filters["defect_size"]:
        allow = {"S", "M", "L", "O"}
        size_wanted = [s for s in filters["defect_size"] if s in allow]
        if not size_wanted:
            size_wanted = None

    # 讀 raw 表快取
    raw_cache: Dict[str, pd.DataFrame] = {}

    def _get_raw(aoi: str, pi: str, ym: str) -> Optional[pd.DataFrame]:
        
        tname = f"{aoi.lower()}_pidensity_{ym}_{pi.lower()}"
        
        if tname not in raw_cache:
            raw_cache[tname] = _try_get_table(dbhandler, tname) or pd.DataFrame()
        return raw_cache[tname]

    # 輸出結構：line_id → aoi → model → glass_type → ai_code_1 → pi_hour → { ... }
    result: Dict[str, Any] = {}

    for _, r in df_filtered.iterrows():
        pi_hour = pd.to_datetime(r["pi_hour"])
        ym = pi_hour.strftime("%Y%m")
        aoi = str(r["aoi"]).upper()
        pi  = str(r["pi"]).upper()

        raw = _get_raw(aoi, pi, ym)
        if raw is None or raw.empty:
            # 試試大寫表名（避免大小寫差異）
            tname2 = f"{aoi}_pidensity_{ym}_{pi}"
            if tname2 not in raw_cache:
                raw_cache[tname2] = _try_get_table(dbhandler, tname2) or pd.DataFrame()
            raw = raw_cache[tname2]
            if raw is None or raw.empty:
                continue

        glass_ids = _sorted_unique_glasses(r.get("glass", ""))

        sub = _filter_raw_for_group(raw, r, glass_ids, size_wanted=size_wanted)
        if sub.empty:
            continue

        # 分 glass 統計 + 缺陷點位
        sub["_bucket"] = sub["defect_size"].apply(_size_bucket)
        by_glass = {}
        for gid, gdf in sub.groupby(sub["glass_id"].astype(str)):
            s = int((gdf["_bucket"] == "S").sum())
            m = int((gdf["_bucket"] == "M").sum())
            l = int((gdf["_bucket"] == "L").sum())
            o = int((gdf["_bucket"] == "O").sum())
            rows = gdf[["x", "y", "chip_name", "pic_name", "defect_size"]].fillna("").to_dict(orient="records")
            by_glass[gid] = {
                "glass_id": gid,
                "glass_defect_count": len(gdf),
                "small_defect_count": s,
                "middle_defect_count": m,
                "large_defect_count": l,
                "over_defect_count": o,
                "rows": rows
            }

        # 塞入巢狀結構
        line_id = r["line_id"]; model = r["model"]; gside = r["glass_type"]
        code    = (r.get("ai_code_1") or "").strip()
        dtype   = (r.get("defect_type") or "").strip()
        ts_key  = pi_hour.strftime("%Y-%m-%d %H:%M:%S")

        d1 = result.setdefault(line_id, {})
        d2 = d1.setdefault(aoi, {})
        d3 = d2.setdefault(model, {})
        d4 = d3.setdefault(gside, {})
        d5 = d4.setdefault(code, {})
        d6 = d5.setdefault(ts_key, {"defect_type": dtype, "glasses": {}})

        # 合併同 group 的 glass（避免同 group 多列）
        for gid, payload in by_glass.items():
            if gid in d6["glasses"]:
                # 合併數量與 rows
                old = d6["glasses"][gid]
                old["glass_defect_count"] += payload["glass_defect_count"]
                old["small_defect_count"]  += payload["small_defect_count"]
                old["middle_defect_count"] += payload["middle_defect_count"]
                old["large_defect_count"]  += payload["large_defect_count"]
                old["over_defect_count"]   += payload["over_defect_count"]
                old["rows"].extend(payload["rows"])
            else:
                d6["glasses"][gid] = payload

    return {
        "DefectGroupDict": result,
        "ParamDict": cfg.front_config
    }